# psxGPT Engine — Course Notebook

Reference implementation of the psxGPT pipeline in Python.
Keep **engine_guide.html** open alongside — the guide explains the architecture; this notebook is where you run it.

---

## Data flow

```
QUERY  ──▶  §1 Router  ──▶  §2 Lanes  ──▶  §3 Post-proc  ──▶  §4 Analyst  ──▶  Answer
                │                │                │                  │
              plan          lane_artifacts   provenanceIds     analyst_result
            (typed asks)   (rows / pages)   (doc IDs)         (text + citations)
```

---

## How to navigate

| Goal | Steps |
|---|---|
| **Run end-to-end (quick start)** | Execute all §0 cells → edit `QUERY_E2E` in §5 → run §5 |
| **Step through stage by stage** | §0 → §1.3b → §2.15 → §3.1 → §4.1 → §4.3 → §4.4 |
| **Inspect one stage** | Re-run just that stage's cells; optional `[OPTIONAL]` cells add debug detail |
| **Look up a function** | Find the matching `§` section in **engine_guide.html** |

---

**Sections:** §0 Setup → §1 Router → §2 Lanes → §3 Post-processing → §4 Analyst → §5 End-to-end → §6 Inspector

---
## §0 — Setup
Run all §0 cells before anything else.

**Produces:** `client` · `run_sql` · `ROUTER_*` / `LANE_*` / `SCHEMA_*` context packs · `TICKERS_WITH_MCAP`

In [ ]:
# ── §0.1  Install dependencies ────────────────────────────────────────────────
# Works in both notebooks and plain Python execution.
import importlib, subprocess, sys

required = ["openai", "requests"]
missing = [m for m in required if importlib.util.find_spec(m) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("Dependency check complete.")


In [ ]:
# ── §0.2  Credentials ─────────────────────────────────────────────────────────
#
#  Three services, three roles:
#    • Gemini Flash Lite  — router decomposition + all lane retrieval (fast, cheap)
#    • DeepSeek V4 Pro    — analyst synthesis (deep reasoning on assembled evidence)
#    • Cloudflare D1      — the financial database (all SQL queries go here via run_sql)
#  Keeping them separate means each can be swapped or upgraded independently.

# Colab-first: keep credentials explicit here (no .dev.vars, no wrangler fallback).

import os

# DeepSeek analyst output
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY", "").strip()

# Gemini retrieval/router/lane
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "").strip()

# Cloudflare D1 REST
CF_ACCOUNT_ID = os.getenv("CF_ACCOUNT_ID", "2e5783bcf42effc537210ebfacff3f56").strip()
CF_DATABASE_ID = os.getenv("CF_DATABASE_ID", "6cd31b68-0737-4d45-a38b-cb225c2f247e").strip()
CF_API_TOKEN = os.getenv("CF_API_TOKEN", "").strip()

if CF_API_TOKEN == CF_ACCOUNT_ID:
    raise ValueError("CF_API_TOKEN is set to the account ID. Paste the API token value (starts with cfat_).")

missing = []
if not GEMINI_API_KEY:
    missing.append("GEMINI_API_KEY")
if not DEEPSEEK_API_KEY:
    missing.append("DEEPSEEK_API_KEY")
if not CF_ACCOUNT_ID:
    missing.append("CF_ACCOUNT_ID")
if not CF_DATABASE_ID:
    missing.append("CF_DATABASE_ID")
if not CF_API_TOKEN or CF_API_TOKEN in {"REPLACE_ME", "cfat_REPLACE_ME"}:
    missing.append("CF_API_TOKEN")

if missing:
    raise ValueError("Set these credentials in §0.2 before running: " + ", ".join(missing))

print("Credentials loaded (values hidden).")
print(f"  GEMINI_API_KEY: {'present' if GEMINI_API_KEY else 'MISSING'}")
print(f"  DEEPSEEK_API_KEY: {'present' if DEEPSEEK_API_KEY else 'MISSING'}")
print("  CF_ACCOUNT_ID: present")
print("  CF_DATABASE_ID: present")
print("  CF_API_TOKEN: present")


In [ ]:
# ── §0.3  Imports and model constants ─────────────────────────────────────────
import json, re, textwrap, datetime, requests
from pathlib import Path
from openai import OpenAI

try:
    from IPython.display import display, Markdown
except Exception:
    def display(x):
        print(x)
    def Markdown(x):
        return x

# Worker-parity model split:
# - Gemini handles router/lane retrieval and page extraction.
# - DeepSeek V4 Pro handles final analyst synthesis with thinking disabled at provider level.
MODEL_ROUTER  = "gemini-3.1-flash-lite-preview"
MODEL_LANE    = "gemini-3.1-flash-lite-preview"
MODEL_ANALYST = "deepseek-v4-pro"

HAS_RETRIEVAL_LLM_KEY = bool(GEMINI_API_KEY)
HAS_ANALYST_LLM_KEY = bool(DEEPSEEK_API_KEY)
retrieval_client = OpenAI(api_key=GEMINI_API_KEY, base_url="https://generativelanguage.googleapis.com/v1beta/openai/") if HAS_RETRIEVAL_LLM_KEY else None
analyst_client = OpenAI(api_key=DEEPSEEK_API_KEY, base_url="https://api.deepseek.com") if HAS_ANALYST_LLM_KEY else None

# Keep the old `client` name for router/lane/extraction cells.
client = retrieval_client

print(f"Models: router={MODEL_ROUTER}, lane/extraction={MODEL_LANE}, analyst={MODEL_ANALYST}")
if not HAS_RETRIEVAL_LLM_KEY:
    print("Warning: GEMINI_API_KEY missing. Router/Lane/Extraction LLM calls will be skipped.")
if not HAS_ANALYST_LLM_KEY:
    print("Warning: DEEPSEEK_API_KEY missing. Analyst LLM calls will be skipped.")


In [ ]:
# ── §0.4  Context pack loader ──────────────────────────────────────────────────
#
#  Loads two categories of content the LLMs will see — distinguished by mutability:
#    FACT        — ground-truth schema files (table definitions, enum values, canonical
#                  name inventories). Wrong if rewritten. Never GEPA candidates.
#    INSTRUCTION — behavioural guidance (routing rules, retrieval playbooks, SQL style).
#                  Shapes how the LLM reasons. Legitimate GEPA optimisation targets.
#  get_lane_pack() (§2.1) and build_router_system() (§1.2) consume these at call time.
#  In:  context/ filesystem  |  Out: SCHEMA_*, ROUTER_*, LANE_* string constants

# Colab-first: expects this notebook inside a folder containing ./context and ./data.


def find_colab_root(start: Path) -> Path:
    for cand in [start, *start.parents]:
        if (cand / "context" / "router" / "fact" / "tables.md").exists() and (cand / "data" / "tickers100.md").exists():
            return cand
    raise FileNotFoundError("Could not find colab root with ./context and ./data")


REPO_ROOT = find_colab_root(Path.cwd().resolve())
CONTEXT_ROOT = REPO_ROOT / "context"
DATA_ROOT = REPO_ROOT / "data"


def load_text(path: Path) -> str:
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    return path.read_text(encoding="utf-8").strip()


def load_json(path: Path) -> dict:
    return json.loads(load_text(path))


ROUTER_TABLES_MD = load_text(CONTEXT_ROOT / "router" / "fact" / "tables.md")
ROUTER_RULES_MD = load_text(CONTEXT_ROOT / "router" / "instruction" / "rules.md")
ROUTER_EXAMPLES_MD = load_text(CONTEXT_ROOT / "router" / "instruction" / "examples.md")
ROUTER_ANALYSIS_INTENTS_MD = load_text(CONTEXT_ROOT / "router" / "instruction" / "analysis_intents.md")
TICKERS_MD = load_text(DATA_ROOT / "tickers100.md")

LANE_FS_MD = load_text(CONTEXT_ROOT / "lane" / "instruction" / "financial_statements.md")
LANE_DOCS_MD = load_text(CONTEXT_ROOT / "lane" / "instruction" / "financial_documents_new.md")
LANE_MARKET_MD = load_text(CONTEXT_ROOT / "lane" / "instruction" / "market_data.md")
LANE_RATIOS_MD = load_text(CONTEXT_ROOT / "lane" / "instruction" / "ratios.md")
LANE_COMP_MD = load_text(CONTEXT_ROOT / "lane" / "instruction" / "management_comp.md")
CANONICALS_JSON = load_json(CONTEXT_ROOT / "lane" / "fact" / "financial_statements_canonicals.json")

# Schema packs — Fact content, never GEPA candidates, always injected verbatim.
SCHEMA_FS_MD = load_text(CONTEXT_ROOT / "lane" / "fact" / "financial_statements.md")
SCHEMA_DOCS_MD = load_text(CONTEXT_ROOT / "lane" / "fact" / "financial_documents_new.md")
SCHEMA_RATIOS_MD = load_text(CONTEXT_ROOT / "lane" / "fact" / "ratios.md")
SCHEMA_COMP_MD = load_text(CONTEXT_ROOT / "lane" / "fact" / "management_comp.md")
SCHEMA_MARKET_MD = load_text(CONTEXT_ROOT / "lane" / "fact" / "market_data.md")
SCHEMA_ROUTER_ASK_MD = load_text(CONTEXT_ROOT / "router" / "fact" / "router_ask.md")
SCHEMA_METRIC_ROUTING_MD = load_text(CONTEXT_ROOT / "router" / "fact" / "metric_routing_index.md")

ANALYST_PROMPT_MD = load_text(CONTEXT_ROOT / "analyst" / "prompt.md")

print("All context packs loaded.")
print(f"Colab root: {REPO_ROOT}")


In [ ]:
# ── §0.5  D1 SQL helper ────────────────────────────────────────────────────────
#
#  run_sql() is the ONLY way the pipeline talks to the database — one function,
#  one responsibility. All lane SQL calls, provenance fetches, and ratio-inventory
#  queries go through here. Validates the CF token at load time so failures surface
#  immediately, not buried in a lane iteration.
#  In:  SQL string + optional params list  |  Out: {success, rows, row_count, columns, error}

# REST-only (no wrangler fallback).

D1_URL = f"https://api.cloudflare.com/client/v4/accounts/{CF_ACCOUNT_ID}/d1/database/{CF_DATABASE_ID}/query"


def run_sql(sql: str, params: list = None) -> dict:
    payload = {"sql": sql}
    if params:
        payload["params"] = params

    try:
        resp = requests.post(
            D1_URL,
            headers={
                "Authorization": f"Bearer {CF_API_TOKEN}",
                "Content-Type": "application/json",
            },
            json=payload,
            timeout=40,
        )
    except Exception as e:
        return {"success": False, "rows": [], "row_count": 0, "columns": [], "error": f"D1 request failed: {e}"}

    if not resp.ok:
        hint = ""
        if resp.status_code == 401:
            hint = " (check CF_API_TOKEN value/scope)"
        return {"success": False, "rows": [], "row_count": 0, "columns": [], "error": f"HTTP {resp.status_code}{hint}: {resp.text[:400]}"}

    data = resp.json()
    if not data.get("success"):
        return {"success": False, "rows": [], "row_count": 0, "columns": [], "error": str(data.get("errors", []))}

    block = (data.get("result") or [{}])[0]
    rows = block.get("results", []) if isinstance(block, dict) else []
    columns = list(rows[0].keys()) if rows else []
    return {"success": True, "rows": rows, "row_count": len(rows), "columns": columns, "error": None}


print(f"CF token loaded: prefix={CF_API_TOKEN[:8]}..., length={len(CF_API_TOKEN)}")
verify_resp = requests.get(
    "https://api.cloudflare.com/client/v4/user/tokens/verify",
    headers={"Authorization": f"Bearer {CF_API_TOKEN}"},
    timeout=20,
)
if not verify_resp.ok:
    raise RuntimeError(f"Cloudflare token verify failed: HTTP {verify_resp.status_code}: {verify_resp.text[:300]}")
verify_payload = verify_resp.json()
if not verify_payload.get("success"):
    raise RuntimeError("Cloudflare token verify failed: " + str(verify_payload.get("errors", [])))
if str((verify_payload.get("result") or {}).get("status", "")).lower() != "active":
    raise RuntimeError("Cloudflare token is not active")
print("Cloudflare token verified: active")

test = run_sql("SELECT date('now') AS today")
if test["success"]:
    print("D1 connected:", test["rows"])
else:
    raise RuntimeError("D1 connection failed: " + str(test["error"]))


In [ ]:
# ── §0.6  Shared utility functions ────────────────────────────────────────────
#
#  Every LLM call (router + all lanes) returns raw text that may contain markdown
#  code fences or extra whitespace. safe_json() strips all of that and returns
#  parsed JSON, or `fallback` if parsing fails — used everywhere downstream.
#  In:  raw model text (str) + a fallback value
#  Out: parsed dict / list, or fallback

def safe_json(text: str, fallback):
    cleaned = re.sub(r'^```(?:json)?\s*', '', text.strip(), flags=re.IGNORECASE)
    cleaned = re.sub(r'\s*```$', '', cleaned)
    try:
        return json.loads(cleaned)
    except Exception:
        m = re.search(r'\{[\s\S]*\}', cleaned)
        if m:
            try:
                return json.loads(m.group(0))
            except Exception:
                pass
    return fallback

print("safe_json ready.")

In [ ]:
# ── §0.7  pp_rows — notebook display helper ──────────────────────────────────────
#
#  Notebook display helper only — NOT used in the pipeline itself.
#  Formats a list of DB row dicts as a markdown table for inline inspection.
#  In:  rows (list[dict])  |  Out: Markdown table rendered in notebook

def pp_rows(rows: list, max_rows: int = 10):
    if not rows:
        display(Markdown("*(no rows)*"))
        return
    cols = list(rows[0].keys())
    header = "| " + " | ".join(cols) + " |"
    sep = "| " + " | ".join(["---"] * len(cols)) + " |"
    body = []
    for row in rows[:max_rows]:
        cells = [str(row.get(c, ""))[:80] for c in cols]
        body.append("| " + " | ".join(cells) + " |")
    md = "\n".join([header, sep] + body)
    if len(rows) > max_rows:
        md += "\n\n" + f"*… {len(rows) - max_rows} more rows*"
    display(Markdown(md))


print("Utilities ready.")

# ✓ Checkpoint — §0 complete
print("\n✓ §0 done: client, run_sql, context packs, and TICKERS_MD are loaded.")
print("  Next: run §1 cells (§1.1 → §1.3b) to run the router and produce plan.")

---
## §1 — Router  ·  `router.ts`

**In:** `QUERY` (string)  →  **Out:** `plan` (list of typed ask dicts)

§1.1–§1.3a define helpers and build the prompt. **§1.3b runs the router and produces `plan`.** §1.4 is `[OPTIONAL]`.

In [ ]:
# ── §1.1  Fetch live market cap to enrich the ticker table ────────────────────
#
#  Builds TICKERS_WITH_MCAP — the router's authoritative ticker universe.
#  Format: Symbol | Company | Industry | MCap (PKR bn). Injected into every router
#  call so the router can scope asks to specific sectors and weight tickers by size.
#  Must run before §1.2 (router prompt assembly).
#  In:  live query to market_data (D1)  |  Out: TICKERS_WITH_MCAP (markdown table string)

def enrich_tickers_with_mcap(tickers_md: str) -> str:
    result = run_sql(
        "WITH latest_per_ticker AS ("
        "  SELECT ticker, MAX(date) AS latest_date "
        "  FROM market_data "
        "  GROUP BY ticker"
        ") "
        "SELECT md.ticker, md.market_cap, md.date "
        "FROM market_data md "
        "JOIN latest_per_ticker lpt "
        "  ON md.ticker = lpt.ticker AND md.date = lpt.latest_date"
    )
    if not result["success"] or not result["rows"]:
        raise RuntimeError("Failed to fetch market caps: " + str(result.get("error")))

    mcap_map = {}
    for row in result["rows"]:
        ticker = str(row.get("ticker", ""))
        mcap = row.get("market_cap") or 0
        mcap_map[ticker] = f"{mcap / 1_000_000_000:.1f}" if mcap else "-"

    lines_out = []
    for line in tickers_md.splitlines():
        if line.startswith("| Symbol"):
            lines_out.append("| Symbol | Company | Industry | MCap(B) |")
        elif line.startswith("| ---"):
            lines_out.append("| --- | --- | --- | --- |")
        elif line.startswith("|"):
            parts = [p.strip() for p in line.split("|") if p.strip()]
            if len(parts) >= 3:
                mcap = mcap_map.get(parts[0], "-")
                lines_out.append(f"| {parts[0]} | {parts[1]} | {parts[2]} | {mcap} |")
        else:
            lines_out.append(line)
    return "\n".join(lines_out)


TICKERS_WITH_MCAP = enrich_tickers_with_mcap(TICKERS_MD)
print("Ticker table enriched. First 5 rows:")
display(Markdown("\n".join(TICKERS_WITH_MCAP.splitlines()[:7])))


In [ ]:
# ── §1.2  Build router system prompt ──────────────────────────────────────────
#
#  Assembles the router's full system prompt from three source categories:
#    FACT:        tables.md (table one-liners) + router_ask.md (output contract,
#                 field enums, ratios inventory) + metric_routing_index.md
#    INSTRUCTION: rules.md + examples.md + analysis_intents.md (always appended)
#    RUNTIME:     live ratios inventory from DB (fresh every session)
#  Context release rule: the router sees its own context only — not lane schemas.
#  In:  TICKERS_WITH_MCAP + loaded context packs  |  Out: ROUTER_SYSTEM (str)

def _extract_ratio_inventory_from_pack(pack_md: str) -> list[str]:
    """Pull the AVAILABLE RATIO NAMES line out of the ratios lane pack."""
    lines = pack_md.splitlines()
    for i, line in enumerate(lines):
        if line.strip().upper().startswith("AVAILABLE RATIO NAMES"):
            if i + 1 < len(lines):
                return [t.strip() for t in lines[i + 1].split(",") if t.strip()]
    return []


def _fetch_live_ratio_inventory() -> list[str]:
    result = run_sql("SELECT DISTINCT ratio_name FROM ratios ORDER BY ratio_name")
    if not result["success"]:
        return []
    return [str(row.get("ratio_name")).strip() for row in result["rows"] if row.get("ratio_name")]


# Resolve the ratios inventory once. Prefer live DB, fall back to pack text.
RATIOS_INVENTORY_LIVE = _fetch_live_ratio_inventory()
RATIOS_INVENTORY_PACK = _extract_ratio_inventory_from_pack(LANE_RATIOS_MD)
RATIOS_INVENTORY = RATIOS_INVENTORY_LIVE or RATIOS_INVENTORY_PACK
print(f"Ratios inventory resolved ({'live' if RATIOS_INVENTORY_LIVE else 'pack'}): {RATIOS_INVENTORY}")

def build_router_system(tickers_with_mcap: str, query_context: str = "") -> str:
    # Fill the ratios inventory slot inside the schema file, then fill the
    # [[SCHEMA: router_ask]] slot in rules.md with the assembled schema block.
    schema_ask_md = SCHEMA_ROUTER_ASK_MD.replace(
        "[[PASTE FROM: SELECT DISTINCT ratio_name FROM ratios ORDER BY ratio_name]]",
        ", ".join(RATIOS_INVENTORY) if RATIOS_INVENTORY else "(inventory unavailable)"
    ).replace("[[SCHEMA: metric_routing_index]]", SCHEMA_METRIC_ROUTING_MD)
    rules_md = ROUTER_RULES_MD.replace("[[SCHEMA: router_ask]]", schema_ask_md)
    sections = [
        "You are the Router for psxGPT. Return JSON only.",
        "",
        "Your job: infer what the user wants, decompose it into concrete asks against specific tables, and produce a query plan.",
        "",
        ROUTER_TABLES_MD,
        "",
        "TICKER REFERENCE",
        tickers_with_mcap,
        "",
        rules_md,
        "",
        ROUTER_EXAMPLES_MD,
    ]
    sections.extend(["", ROUTER_ANALYSIS_INTENTS_MD])
    return "\n".join(sections)


ROUTER_SYSTEM = build_router_system(TICKERS_WITH_MCAP)
print(f"Router base system prompt built: {len(ROUTER_SYSTEM):,} chars")
print("Analysis-intent router pack appended unconditionally.")


In [ ]:
# ── §1.3a  Router ask contract & output normalisers ────────────────────────────
#
#  The router LLM returns raw JSON that may have typos, invalid tables, or unknown
#  tickers. These functions define the TYPED ASK CONTRACT and enforce it on raw output.
#    • VALID_TABLES / VALID_STATEMENT_TYPES — closed-vocabulary constants (the contract)
#    • normalize_router_ask() — main normaliser: validates, repairs, returns ask or None
#    • _repair_ticker, _normalize_string_list, format_ask_* — building-block helpers
#  In:  raw LLM JSON item (dict)  |  Out: validated ask dict, or None if invalid

# Debug-first router call. No silent fallbacks.

VALID_TABLES = {
    "financial_statements",
    "financial_documents_new",
    "management_comp",
    "market_data",
    "ratios",
}
VALID_STATEMENT_TYPES = {"profit_loss", "balance_sheet", "cash_flow"}
VALID_POSTURES = {"narrative", "numeric"}

import difflib


def _parse_ticker_reference(md: str) -> dict[str, str]:
    out = {}
    for line in str(md or "").splitlines():
        if not line.startswith("|") or "---" in line or "Symbol" in line:
            continue
        parts = [p.strip() for p in line.strip("|").split("|")]
        if len(parts) >= 3 and re.fullmatch(r"[A-Z0-9]{2,8}", parts[0]):
            out[parts[0].upper()] = parts[2]
    return out


TICKER_INDUSTRY_MAP = _parse_ticker_reference(TICKERS_MD)
VALID_TICKERS = set(TICKER_INDUSTRY_MAP)


def _industry_tickers(industry: str) -> list[str]:
    target = str(industry or "").strip().lower()
    return [t for t, ind in TICKER_INDUSTRY_MAP.items() if str(ind).strip().lower() == target]


def _repair_ticker(ticker: str, industry: str = "") -> str | None:
    ticker = str(ticker or "").strip().upper()
    if ticker in VALID_TICKERS:
        return ticker
    choices = _industry_tickers(industry) if industry else sorted(VALID_TICKERS)
    match = difflib.get_close_matches(ticker, choices, n=1, cutoff=0.78)
    return match[0] if match else None


def _normalize_string_list(values) -> list:
    if values is None:
        return []
    if isinstance(values, str):
        values = [values]
    out = []
    for value in values:
        text = str(value).strip()
        if text:
            out.append(text)
    return list(dict.fromkeys(out))


def normalize_router_ask(item: dict) -> dict | None:
    table = item.get("table")
    ask = str(item.get("ask", "")).strip()
    if table not in VALID_TABLES or not ask:
        return None

    industry = str(item.get("industry", "")).strip()

    # ── Typed scope field replaces prose-regex scope exceptions ───────────────
    # The router emits scope="market_wide" for cross-universe screens/rankings
    # (no ticker filter, lane does full-table scan) and scope="universe" for
    # count/list-all asks on market_data. Both bypass the entity-scope requirement.
    scope = str(item.get("scope") or "").strip().lower()
    # scope="universe" is only valid for market_data (count/list-all asks).
    # If another table emits it, treat it as an invalid scope and clear it so
    # the ask either resolves via tickers/industry or becomes _missing_scope.
    if scope == "universe" and table != "market_data":
        scope = ""
    _is_scopeless_ok = scope in {"market_wide", "universe"}

    # ── Ticker resolution ──────────────────────────────────────────────────────
    raw_tickers = _normalize_string_list(item.get("tickers"))
    tickers = []
    unresolved_tickers = []
    for value in raw_tickers:
        repaired = _repair_ticker(value, industry)
        if repaired:
            tickers.append(repaired)
        else:
            unresolved_tickers.append(value.strip().upper())
    tickers = list(dict.fromkeys(tickers))

    # If industry is present and no specific tickers resolved, expand to the full
    # industry universe. When the router emits both industry and specific tickers[]
    # (for traceability), trust the router's tickers — do not over-expand.
    if industry and not tickers:
        expanded = _industry_tickers(industry)
        if expanded:
            tickers = expanded

    # ── Out-of-universe detection ──────────────────────────────────────────────
    # Fire when ANY router-provided tickers failed to resolve (not only when ALL fail).
    # This surfaces partial drops (e.g. AIRLINK missing from the coverage universe
    # while other tickers in the same ask resolved fine).
    if unresolved_tickers and not _is_scopeless_ok:
        if not tickers and not industry:
            # All tickers unresolved → full OOU short-circuit
            return {
                "table": table,
                "ask": ask,
                "tickers": [],
                "_out_of_universe": unresolved_tickers,
            }
        # Some resolved, some not → proceed with resolved tickers but flag the drop
        # so the analyst context can surface it as a partial coverage gap.

    # ── Missing-scope guard ────────────────────────────────────────────────────
    if not tickers and not industry and not _is_scopeless_ok:
        return {
            "table": table,
            "ask": ask,
            "tickers": [],
            "_missing_scope": "company/ticker or industry scope is required for this ask",
        }

    ask_obj = {"table": table, "ask": ask}
    if tickers:
        ask_obj["tickers"] = tickers
    if industry:
        ask_obj["industry"] = industry
    if scope:
        ask_obj["scope"] = scope
    if item.get("is_selector"):
        ask_obj["is_selector"] = True
    if unresolved_tickers:
        ask_obj["_unresolved_tickers"] = unresolved_tickers

    if table == "financial_statements":
        statement_types = [
            value
            for value in _normalize_string_list(item.get("statement_types"))
            if value in VALID_STATEMENT_TYPES
        ]
        if statement_types:
            ask_obj["statement_types"] = statement_types

    if table == "financial_documents_new":
        posture = str(item.get("posture", "")).strip().lower()
        concepts = _normalize_string_list(item.get("concepts"))
        if posture in VALID_POSTURES:
            ask_obj["posture"] = posture
        if concepts:
            ask_obj["concepts"] = concepts[:20]

    if table == "ratios":
        ratio_names = [
            value.strip().lower()
            for value in _normalize_string_list(item.get("ratio_names"))
            if value.strip().lower() in {r.lower() for r in RATIOS_INVENTORY}
        ]
        ratio_names = list(dict.fromkeys(ratio_names))
        if ratio_names:
            ask_obj["ratio_names"] = ratio_names

    return ask_obj


def format_ask_scope(ask_obj: dict) -> str:
    parts = []
    if ask_obj.get("scope"):
        parts.append(f"scope={ask_obj['scope']}")
    if ask_obj.get("tickers"):
        parts.append("tickers=" + ",".join(ask_obj["tickers"]))
    if ask_obj.get("industry"):
        parts.append(f"industry={ask_obj['industry']}")
    if ask_obj.get("is_selector"):
        parts.append("is_selector=true")
    if ask_obj.get("_unresolved_tickers"):
        parts.append("unresolved=" + ",".join(ask_obj["_unresolved_tickers"]))
    if ask_obj.get("statement_types"):
        parts.append("statement_types=" + ",".join(ask_obj["statement_types"]))
    if ask_obj.get("posture"):
        parts.append(f"posture={ask_obj['posture']}")
    if ask_obj.get("concepts"):
        parts.append(f"concepts={len(ask_obj['concepts'])}")
    if ask_obj.get("ratio_names"):
        parts.append("ratio_names=" + ",".join(ask_obj["ratio_names"]))
    return "; ".join(parts)


def format_ask_line(ask_obj: dict) -> str:
    scope = format_ask_scope(ask_obj)
    suffix = f" [{scope}]" if scope else ""
    return f"[{ask_obj['table']}] {ask_obj['ask']}{suffix}"

print("Router normalisers ready.")

In [ ]:
# ── §1.3b  run_router — router orchestration ────────────────────────────────────
#
#  Orchestrates the router stage end-to-end:
#    1. Call Gemini with ROUTER_SYSTEM + query
#    2. Parse the JSON response via safe_json()
#    3. Normalise each ask with normalize_router_ask() from §1.3a
#    4. Apply scope guards (out-of-universe detection, missing-scope)
#    5. Return plan — the typed asks the lanes will execute next
#  In:  QUERY (str)  |  Out: plan dict  →  plan['asks'] list of typed ask objects

def run_router(query: str) -> dict:
    if client is None:
        raise RuntimeError("Gemini retrieval client unavailable. Set GEMINI_API_KEY in §0.2.")

    today = datetime.date.today().isoformat()
    user_message = f"[Today: {today}] {query}"

    system_prompt = build_router_system(TICKERS_WITH_MCAP, user_message)

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message},
    ]

    response = client.chat.completions.create(
        model=MODEL_ROUTER,
        messages=messages,
        temperature=0.1,
    )

    raw = response.choices[0].message.content or ""
    parsed = safe_json(raw, {"asks": []})

    asks = []
    for item in parsed.get("asks", []):
        ask_obj = normalize_router_ask(item)
        if ask_obj:
            asks.append(ask_obj)

    return {
        "plan": asks[:8],
        "system_prompt": system_prompt,
        "user_message": user_message,
        "messages": messages,
        "raw_content": raw,
        "usage": {
            "input_tokens": response.usage.prompt_tokens,
            "output_tokens": response.usage.completion_tokens,
        },
        "parse_ok": len(asks) > 0,
    }


# ── Try it ─────────────────────────────────────────────────────────────────────
QUERY = "How has UBL's deposit strategy evolved over the last 4 years, and what do the numbers show?"

router_result = run_router(QUERY)
plan = router_result["plan"]

print(f"Router produced {len(plan)} ask(s).  "
      f"Tokens: {router_result['usage']['input_tokens']} in / "
      f"{router_result['usage']['output_tokens']} out")

if not router_result["parse_ok"]:
    print("Warning: router JSON parse produced zero valid asks. Inspect §1.4 output.")

for i, ask in enumerate(plan, 1):
    print(f"  Ask {i}: {format_ask_line(ask)}")

# ✓ Checkpoint — §1 complete
print(f"\n✓ §1 done: plan has {len(plan)} ask(s). Next: run §2 cells (§2.1–§2.15).")

In [ ]:
# ── §1.4  Inspect router internals (optional) ─────────────────────────────────
# Run this to see exactly what the router was sent and what it returned raw.

print("=== SYSTEM PROMPT ===")
print(router_result.get("system_prompt", "(missing)"))

print()
print("=== USER MESSAGE ===")
print(router_result.get("user_message", "(missing)"))

print()
print("=== FULL MODEL INPUT MESSAGES (JSON) ===")
print(json.dumps(router_result.get("messages", []), ensure_ascii=False, indent=2))

print()
print("=== RAW MODEL RESPONSE ===")
print(router_result.get("raw_content", ""))


---
## §2 — Lanes  ·  `lane.ts`

**In:** ask dict from `plan`  →  **Out:** `lane_artifacts` (list of artifact dicts)

§2.1–§2.14 define all lane helpers and runners (run them in order — each builds on the previous).
**§2.15 executes the full lane sequence and produces `lane_artifacts`.**
§2.16 is `[OPTIONAL]`.

In [ ]:
# ── §2.1  Build lane system prompts ─────────────────────────────────────────────────
#
#  Each lane LLM only sees ITS OWN table's context — not other tables' schemas.
#  This is the context-release rule: information released only to the component
#  that can act on it. get_lane_pack() assembles FACT + INSTRUCTION for one table;
#  build_lane_system() adds the role header, lane completion contract, and ask object.
#  In:  table name + ask_obj (typed ask from router)
#  Out: system prompt string for this lane's LLM call

# Mirrors buildLanePrompt() in context.ts.

_INDUSTRY_TO_CANONICAL_FAMILY = {'banking': 'BANK', 'insurance': 'INSURANCE'}


def canonical_families_for_tickers(tickers: list[str]) -> list[str]:
    families = set()
    for ticker in tickers:
        industry = TICKER_INDUSTRY_MAP.get(ticker.upper(), '').lower()
        families.add(_INDUSTRY_TO_CANONICAL_FAMILY.get(industry, 'CORPORATE'))
    return sorted(families) if families else list(CANONICALS_JSON.keys())


def _fill_schema_slot(instruction_md: str, table_name: str, schema_md: str) -> str:
    """Replace [[SCHEMA: table_name]] with the verbatim schema content."""
    return instruction_md.replace(f"[[SCHEMA: {table_name}]]", schema_md)


def get_lane_pack(table: str, tickers: list[str] | None = None) -> str:
    if table == 'financial_statements':
        families = canonical_families_for_tickers(tickers or [])
        filtered = {k: v for k, v in CANONICALS_JSON.items() if k in families}
        instructions = _fill_schema_slot(LANE_FS_MD, 'financial_statements', SCHEMA_FS_MD)
        return '\n'.join([
            instructions,
            '',
            f"CANONICAL REFERENCE ({', '.join(families)} \u2014 matched to ticker industry):",
            '```json',
            json.dumps(filtered, indent=2),
            '```',
        ])
    elif table == 'financial_documents_new':
        return _fill_schema_slot(LANE_DOCS_MD, 'financial_documents_new', SCHEMA_DOCS_MD)
    elif table == 'market_data':
        return _fill_schema_slot(LANE_MARKET_MD, 'market_data', SCHEMA_MARKET_MD)
    elif table == 'ratios':
        return _fill_schema_slot(LANE_RATIOS_MD, 'ratios', SCHEMA_RATIOS_MD)
    elif table == 'management_comp':
        return _fill_schema_slot(LANE_COMP_MD, 'management_comp', SCHEMA_COMP_MD)
    else:
        return f'Table: {table} \u2014 no pack available.'


STRUCTURED_LANE_CONTRACT = [
    "LANE COMPLETION CONTRACT (structured lane)",
    "- You must call `submit` exactly once before the lane ends.",
    "- If the typed fields on the ask already name the scope (tickers, concepts, ratio_names, statement_types), use them directly. Do not run discovery queries to re-derive information the router already provided.",
    "- Discovery SQL is allowed only to unblock genuine uncertainty the typed ask does not resolve. Keep discovery to at most one query.",
    "- Early-exit rule: once the current query rows already answer the ask, call `submit` on the same turn. Do not run another query to refine, re-probe, or reformulate.",
    "- Once you have enough evidence, run the final analytical SQL and then call `submit` in the same turn.",
    "- For selector/ranking/filter asks, include `selected_tickers` in submit when the result establishes a narrowed entity set for later lanes. Do not include it for ordinary compare/trend asks.",
    "- Do not spend iterations on DISTINCT or probe queries when the ask is already scoped.",
    "- On the final iteration, submit only if the result is valid. Otherwise the lane fails.",
    "- A lane that ends without a valid submit is a failure.",
    "",
    "SUBMIT REJECTION CONDITIONS (structured lane)",
    "- Submit is rejected if `final_sql` matches a discovery pattern (SELECT DISTINCT of a single field, COUNT(*) + GROUP BY, SELECT *), or if the final rows carry only metadata columns without an analytical value column.",
    "- A rejected submit does not end the lane \u2014 the tool returns an error and the iteration continues.",
    "",
    "Your final SQL must answer the ask \u2014 not merely list tickers, periods, or field names.",
    "Before calling submit, execute your final analytical SQL with run_sql and confirm the rows answer the ask.",
]

PAGE_SELECTION_LANE_CONTRACT = [
    "LANE COMPLETION CONTRACT (page-selection lane)",
    "- You must call `submit` exactly once before the lane ends.",
    "- This lane returns page IDs, not aggregated rows. There is NO \"final analytical SQL\" for this lane \u2014 the candidate set produced by FTS retrieval (optionally refined by shaping tools) IS the answer.",
    "- If the typed fields on the ask already name the scope (tickers, concepts, filing_type), use them directly. Do not run discovery queries for filing_period, ticker counts, or page counts.",
    "- Normal flow: (a) one FTS retrieval query using concepts as MATCH terms, (b) optional shaping tool(s) to balance or refine the candidate set, (c) submit with the selected page IDs.",
    "- Early-exit rule: once the candidate set covers every ticker in the ask with readable pages, call `submit` on the same turn. Do not call another shaping tool, another retrieval, or another query to further refine \u2014 one more shape is not better than submit.",
    "- Do NOT attempt to query `score` from the base table in a follow-up SQL \u2014 `score` only exists inside the FTS retrieval query itself.",
    "- On the final iteration, submit the current candidate set if it is valid. Do not run another retrieval or shaping call on the final iteration \u2014 use it to submit.",
    "- A lane that ends without a valid submit is a failure.",
    "",
    "SUBMIT REJECTION CONDITIONS (page-selection lane)",
    "- Submit is rejected if `source_ids[]` (explicit or inferred from the candidate set) does not include at least one page per ticker named in the ask.",
    "- Submit is rejected if the candidate set is empty.",
    "- A rejected submit does not end the lane \u2014 the tool returns an error and the iteration continues.",
]



def lane_execution_ask_object(ask_obj: dict) -> dict:
    """Return only execution-relevant ask fields for the lane prompt."""
    return {k: v for k, v in ask_obj.items() if not str(k).startswith("_")}

def build_lane_system(table: str, ask_obj: dict) -> str:
    tickers = [str(t).upper() for t in ask_obj.get('tickers', []) if str(t).strip()]
    pack = get_lane_pack(table, tickers)
    execution_ask = lane_execution_ask_object(ask_obj)
    ask_json = json.dumps(execution_ask, indent=2, ensure_ascii=True)
    is_page_selection = (table == 'financial_documents_new')
    contract_block = PAGE_SELECTION_LANE_CONTRACT if is_page_selection else STRUCTURED_LANE_CONTRACT
    lane_kind_line = (
        'This is a page-selection lane. You retrieve and shape pages; you do not run a final analytical SQL.'
        if is_page_selection else
        'This is a structured analytical lane. Write SQL, inspect results, iterate if needed, then call submit.'
    )
    return '\n'.join([
        f'You are a Lane Executor for psxGPT. You query one table: {table}.',
        lane_kind_line,
        'Typed fields constrain retrieval. Use only the tickers listed in ASK OBJECT tickers[]. Do not infer or add ticker symbols from the original user wording or diagnostic fields.',
        '',
        *contract_block,
        '',
        'TABLE CONTEXT',
        pack,
        '',
        'ASK OBJECT',
        '```json',
        ask_json,
        '```',
        '',
        'ASK PURPOSE',
        ask_obj['ask'],
    ])


print("Lane prompt builders ready.")

In [ ]:
# ── §2.2  Tool definitions ─────────────────────────────────────────────────────
#
#  Lanes are LLM agents — they interact with the DB via tools, not free-form text.
#    Structured lanes:  run_sql + submit only
#    Narrative lanes:   run_sql + submit + shaping tools + extract_from_pages
#  Tools define the lane's action space: the LLM can only do what a tool allows.
#  In:  (static definitions)  |  Out: TOOL_RUN_SQL, TOOL_SUBMIT, SHAPING_TOOLS, TOOL_EXTRACT_FROM_PAGES

# Mirrors the tool definitions in lane.ts.
# Structured lanes get run_sql + submit.
# Narrative lanes additionally get shaping tools + extract_from_pages.


TOOL_RUN_SQL = {
    "type": "function",
    "function": {
        "name": "run_sql",
        "description": "Execute a read-only SQL query against the assigned table. Pass keep_ids to preserve specific pages from the current candidate set into inventory before this retrieval replaces the working set.",
        "parameters": {
            "type": "object",
            "properties": {
                "sql": {"type": "string"},
                "keep_ids": {
                    "type": "array",
                    "items": {"type": "number"},
                    "description": "Page IDs from the current candidate set worth keeping. They are merged with this retrieval's results.",
                },
            },
            "required": ["sql"],
        },
    },
}

TOOL_SUBMIT = {
    "type": "function",
    "function": {
        "name": "submit",
        "description": "Submit the final result with a title, final SQL, optional source_ids for narrative lanes, and optional selected_tickers for selector lanes.",
        "parameters": {
            "type": "object",
            "properties": {
                "title":    {"type": "string"},
                "final_sql": {"type": "string"},
                "source_ids": {"type": "array", "items": {"type": "number"}},
                "selected_tickers": {"type": "array", "items": {"type": "string"}},
            },
            "required": ["title"],
        },
    },
}

SHAPING_TOOLS = [
    {"type": "function", "function": {
        "name": "dedupe_adjacent",
        "description": "Remove near-duplicate adjacent pages from the candidate set.",
        "parameters": {"type": "object", "properties": {}},
    }},
    {"type": "function", "function": {
        "name": "section_cue_bias",
        "description": "Boost pages matching section cues such as directors report or chairman review.",
        "parameters": {
            "type": "object",
            "properties": {"cues": {"type": "array", "items": {"type": "string"}}},
        },
    }},
    {"type": "function", "function": {
        "name": "top_k_per_ticker",
        "description": "Cap candidates at N per ticker.",
        "parameters": {
            "type": "object",
            "properties": {"max": {"type": "number"}},
        },
    }},
    {"type": "function", "function": {
        "name": "expand_neighbors",
        "description": "Fetch pages within +/-radius of the best anchor page per ticker.",
        "parameters": {
            "type": "object",
            "properties": {"radius": {"type": "number"}},
        },
    }},
    {"type": "function", "function": {
        "name": "same_filing_focus",
        "description": "Re-sort to concentrate on the dominant filing per ticker.",
        "parameters": {"type": "object", "properties": {}},
    }},
]

TOOL_EXTRACT_FROM_PAGES = {
    "type": "function",
    "function": {
        "name": "extract_from_pages",
        "description": "Extract compact query-shaped text from candidate filing pages. Use after retrieval and shaping for compare/evolve/characterize asks.",
        "parameters": {
            "type": "object",
            "properties": {
                "page_ids": {"type": "array", "items": {"type": "number"}},
                "extraction_prompt": {"type": "string"},
            },
            "required": ["page_ids", "extraction_prompt"],
        },
    },
}

SHAPING_TOOL_NAMES = {t["function"]["name"] for t in SHAPING_TOOLS}
NULL_MARKERS = {"null", "not_present", "not present", "none", "n/a"}

print("Tool definitions ready.")

In [ ]:
# ── §2.3  Narrative row normalization ─────────────────────────────────────
#
#  The narrative lane queries financial_documents_new and gets raw SQL rows.
#  normalize_narrative_row() converts each row into a candidate dict with consistent
#  fields (id, ticker, score, filing metadata). Candidates are the unit of currency
#  for the shaping tools (§2.5) and the balanced-selection logic (§2.9).
#  In:  raw SQL row dict  |  Out: candidate dict {id, ticker, score, filing_period, …}

# Narrative lanes keep one active candidate workspace.

MAX_CANDIDATES_PER_TICKER = 8
NEIGHBOR_EXPANSION_RADIUS = 3
SECTION_CUES = [
    "directors' report",
    "directors report",
    "chairman's review",
    "chairman review",
    "ceo review",
    "ceo reflection",
    "management discussion",
    "management commentary",
    "future outlook",
    "near-term outlook",
    "principal risks",
]

def normalize_text_for_compare(text: str) -> str:
    return re.sub(r"\s+", " ", re.sub(r"[%/()+:;,.']", " ", text.lower())).strip()


def normalize_narrative_rows(rows: list) -> list:
    candidates = []
    for row in rows:
        pid = row.get("id")
        if not isinstance(pid, int):
            continue
        candidates.append({
            "id": pid,
            "ticker": str(row.get("ticker") or ""),
            "filing_type": str(row.get("filing_type") or ""),
            "filing_period": str(row.get("filing_period") or ""),
            "filing_year": str(row.get("filing_year") or ""),
            "pg": int(row.get("pg") or 0),
            "search_text": str(row.get("search_text") or ""),
            "score": row.get("score") if isinstance(row.get("score"), (int, float)) else None,
        })
    return candidates


In [ ]:
# ── §2.4  Narrative retrieval execution ────────────────────────────────────
#
# NOTE: This cell is a section-header placeholder only. No functions are defined here.
#
# The former execute_narrative_retrieval_sql(sql, ask_obj) wrapper was removed
# because it was a one-line pass-through with no added value:
#
#     def execute_narrative_retrieval_sql(sql, ask_obj):
#         return run_sql(sql)              # ← literally all it did
#
# Its only call site is now inlined directly in run_narrative_lane (Cell 29 / §2.13):
#
#     result = run_sql(sql)
#
# Balanced multi-ticker retrieval is the lane's responsibility via UNION ALL SQL.
# Coverage validation happens deterministically at submit time (Cell 24 / §2.9).

In [ ]:
# ── §2.5  Narrative candidate ranking and shaping ─────────────────────
#
#  After the first SQL pass the lane may have 100+ candidate pages.
#  Shaping tools let the lane LLM narrow, re-rank, and deduplicate candidates
#  before deciding which pages to extract and submit — improving evidence quality
#  without burning extra SQL round-trips.
#  In:  candidates list (from §2.3 normalisation)  |  Out: reshaped candidates list

def rank_candidates(candidates: list) -> list:
    return sorted(
        candidates,
        key=lambda c: (
            c.get("ticker", ""),
            0 if c.get("filing_type") == "annual" else 1,
            -(int(str(c.get("filing_period") or "0").replace("-", ""))),
            c.get("score") if c.get("score") is not None else 1e9,
            c.get("pg") or 0,
        ),
    )


def summarize_candidates(candidates: list, limit: int = 8) -> list:
    summary = []
    for c in rank_candidates(candidates)[:limit]:
        summary.append({
            "id": c["id"],
            "ticker": c["ticker"],
            "filing_period": c["filing_period"],
            "pg": c["pg"],
            "score": c.get("score"),
        })
    return summary


def execute_shaping_tool(tool_name: str, args: dict, candidates: list) -> list:
    if tool_name == "dedupe_adjacent":
        kept = []
        for candidate in sorted(candidates, key=lambda c: (c["ticker"], c["filing_period"], c["pg"])):
            prev = kept[-1] if kept else None
            if (
                prev
                and prev["ticker"] == candidate["ticker"]
                and prev["filing_period"] == candidate["filing_period"]
                and abs(prev["pg"] - candidate["pg"]) <= 1
                and normalize_text_for_compare(prev["search_text"]) == normalize_text_for_compare(candidate["search_text"])
            ):
                prev_score = prev.get("score") if prev.get("score") is not None else 1e9
                candidate_score = candidate.get("score") if candidate.get("score") is not None else 1e9
                if candidate_score < prev_score:
                    kept[-1] = candidate
                continue
            kept.append(candidate)
        return rank_candidates(kept)

    if tool_name == "section_cue_bias":
        cues = [str(c).lower() for c in args.get("cues", [])] or SECTION_CUES
        biased = []
        for candidate in candidates:
            lower = candidate["search_text"].lower()
            if any(cue in lower for cue in cues):
                score = candidate.get("score") if candidate.get("score") is not None else 0
                biased.append({**candidate, "score": score - 12})
            else:
                biased.append(candidate)
        return rank_candidates(biased)

    if tool_name == "top_k_per_ticker":
        k = int(args.get("max") or MAX_CANDIDATES_PER_TICKER)
        counts, out = {}, []
        for candidate in rank_candidates(candidates):
            ticker = candidate["ticker"]
            if counts.get(ticker, 0) >= k:
                continue
            out.append(candidate)
            counts[ticker] = counts.get(ticker, 0) + 1
        return out

    if tool_name == "same_filing_focus":
        latest_by_ticker = {}
        for candidate in candidates:
            ticker = candidate.get("ticker")
            period = str(candidate.get("filing_period") or "")
            if not ticker or not period:
                continue
            latest_by_ticker[ticker] = max(latest_by_ticker.get(ticker, ""), period)
        focused = [
            candidate for candidate in candidates
            if not latest_by_ticker.get(candidate.get("ticker"))
            or str(candidate.get("filing_period") or "") == latest_by_ticker[candidate.get("ticker")]
        ]
        return rank_candidates(focused or candidates)

    if tool_name == "expand_neighbors":
        radius = int(args.get("radius") or NEIGHBOR_EXPANSION_RADIUS)
        anchors_by_ticker = {}
        for candidate in rank_candidates(candidates):
            anchors_by_ticker.setdefault(candidate["ticker"], []).append(candidate)
        anchors = []
        for bucket in anchors_by_ticker.values():
            anchors.extend(bucket[:2])

        merged = {c["id"]: c for c in candidates}
        for anchor in anchors:
            start_pg = max(1, anchor["pg"] - radius)
            end_pg = anchor["pg"] + radius
            result = run_sql(
                "SELECT id, ticker, filing_type, filing_period, filing_year, pg, search_text "
                "FROM financial_documents_new "
                "WHERE ticker = ? AND filing_type = ? AND filing_period = ? AND pg BETWEEN ? AND ? "
                "ORDER BY pg",
                [anchor["ticker"], anchor["filing_type"], anchor["filing_period"], start_pg, end_pg],
            )
            for row in result.get("rows", []):
                pid = row.get("id")
                if not isinstance(pid, int):
                    continue
                merged.setdefault(pid, {
                    "id": pid,
                    "ticker": str(row.get("ticker") or ""),
                    "filing_type": str(row.get("filing_type") or ""),
                    "filing_period": str(row.get("filing_period") or ""),
                    "filing_year": str(row.get("filing_year") or ""),
                    "pg": int(row.get("pg") or 0),
                    "search_text": str(row.get("search_text") or ""),
                    "score": anchor.get("score"),
                })
        return rank_candidates(list(merged.values()))

    return rank_candidates(candidates)

print("Narrative candidate helpers ready.")


In [ ]:
# ── §2.6  extract_from_pages ───────────────────────────────────────────────────
#
#  Narrative lanes find relevant PAGES via BM25, then extract EVIDENCE from them.
#  extract_from_pages() fetches page text from D1 and calls Gemini to pull structured
#  snippets using the lane's own extraction prompt. Two-stage design keeps retrieval
#  (BM25 keyword match) separate from extraction (semantic understanding).
#  In:  page IDs list + ask_obj (for extraction prompt framing)
#  Out: list of {page_id, extracted_text} dicts → stored in narrativeExtractions

# Runs one sub-call per selected page after retrieval + shaping.
# The return-format spec is always appended to the user message regardless of
# what extraction_prompt the lane LLM wrote — otherwise the page-LLM invents
# its own JSON shape and our parser drops everything as null.

MAX_EXTRACTION_PAGES = 24

EXTRACTION_RETURN_SPEC = "\n".join([
    "Return JSON only with this exact shape:",
    'If the page does not discuss the target topic: {"extracted_text": null}',
    'If relevant content exists: {"extracted_text": "<one compact paragraph with characterization and supporting quote>"}',
    "Do not include any additional keys.",
])


def build_default_extraction_prompt(ask: str) -> str:
    return "\n".join([
        f"Task: {ask}",
        "Extract only information that directly answers the task from this single filing page.",
        EXTRACTION_RETURN_SPEC,
    ])


def extract_from_pages(page_ids: list, extraction_prompt: str) -> list:
    ids = list(dict.fromkeys(int(i) for i in page_ids if int(i) > 0))[:MAX_EXTRACTION_PAGES]
    if not ids:
        return []

    placeholders = ",".join(["?"] * len(ids))
    rows_result = run_sql(
        f"SELECT id, text, search_text FROM financial_documents_new WHERE id IN ({placeholders})",
        ids,
    )
    row_map = {row["id"]: row for row in rows_result.get("rows", [])}

    extractions = []
    for i, pid in enumerate(ids):
        row = row_map.get(pid)
        if not row:
            continue
        page_text = str(row.get("text") or row.get("search_text") or "").strip()
        if not page_text:
            continue

        print(f"  Extracting page {pid} ({i + 1}/{len(ids)})...", end=" ")
        user_message = "\n".join([
            extraction_prompt,
            "",
            EXTRACTION_RETURN_SPEC,
            "",
            f"PAGE_ID: {pid}",
            "PAGE_TEXT_START",
            page_text[:6000],
            "PAGE_TEXT_END",
        ])
        try:
            resp = client.chat.completions.create(
                model=MODEL_LANE,
                messages=[
                    {"role": "system", "content": "You extract query-targeted evidence from one filing page. Return JSON only."},
                    {"role": "user", "content": user_message},
                ],
                temperature=0.1,
                response_format={"type": "json_object"},
                timeout=30,
            )
        except Exception as _exc:
            print(f"timeout/error: {_exc}")
            continue
        raw = resp.choices[0].message.content or ""
        parsed = safe_json(raw, {"extracted_text": None})
        extracted = str(parsed.get("extracted_text") or "").strip()
        is_null = not extracted or extracted.lower() in NULL_MARKERS
        print("null" if is_null else f"{len(extracted)} chars")
        if not is_null:
            extractions.append({"page_id": pid, "extracted_text": extracted})

    return extractions

print(f"extract_from_pages ready (cap={MAX_EXTRACTION_PAGES} pages).")


In [ ]:
# ── §2.7  Runtime helper primitives ─────────────────────────────────────
#
#  Shared primitives consumed by both lane runners:
#    • MAX_LANE_ITERATIONS — each lane gets ≤ 4 LLM turns to run_sql then submit
#    • _unique_nonempty, _parse_iso_date, _date_gap_days — data-cleaning utilities
#    • _build_table_attachment — packages SQL rows into the analysisAttachment format
#      the analyst will receive (title, headers, rows, coverage metadata)
#  In:  varies per helper  |  Out: cleaned values / attachment dict / constant

# Separate structured + narrative runners keep the notebook logic easier to inspect.

MAX_LANE_ITERATIONS = 4
MAX_SELECTED_SOURCE_IDS = 12
MAX_SQL_RESULT_SAMPLE_ROWS = 12


def _unique_nonempty(values: list) -> list:
    out = []
    for value in values:
        clean = str(value or "").strip()
        if clean:
            out.append(clean)
    return list(dict.fromkeys(out))


def _parse_iso_date(value: str):
    from datetime import datetime as _datetime
    text = str(value or "").strip()
    if not text:
        return None
    try:
        return _datetime.fromisoformat(text[:10]).date()
    except Exception:
        return None


def _date_gap_days(values: list[str]) -> int | None:
    dates = [_parse_iso_date(v) for v in values]
    dates = [d for d in dates if d]
    if len(dates) < 2:
        return None
    return (max(dates) - min(dates)).days


# ── output formatter used by run_structured_lane (§2.12) ────────────────────
def _build_table_attachment(variable: str, title: str, rows: list, table: str = "", ask_obj: dict | None = None, row_flow: dict | None = None, selected_tickers: list | None = None):
    if not rows:
        return None
    headers = list(rows[0].keys())
    rows_data = [[row.get(h) for h in headers] for row in rows]
    csv_lines = [",".join(str(v) for v in headers)]
    for row in rows_data:
        csv_lines.append(",".join(str(v) if v is not None else "" for v in row))
    coverage = _structured_row_coverage(table, ask_obj or {}, rows, row_flow=row_flow, selected_tickers=selected_tickers)
    summary = f"{len(rows)} rows"
    coverage_summary = _coverage_summary_text(coverage)
    if coverage_summary:
        summary += f"; {coverage_summary}"
    return {
        "id": variable,
        "type": "table",
        "title": title,
        "headers": headers,
        "rows": rows_data,
        "csv": "\n".join(csv_lines),
        "summary": summary,
        "coverage": coverage,
    }


In [ ]:
# ── §2.8  Evidence coverage helpers ───────────────────────────────────
#
#  Coverage is a first-class output. Every structured lane reports whether its SQL
#  results covered the requested tickers and periods. This metadata travels with the
#  data to the analyst — gaps are surfaced explicitly, never silently omitted.
#  In:  table, ask_obj, rows (from run_sql)
#  Out: coverage dict  →  {requested_tickers, fed_tickers, missing_tickers, periods_present, …}

def _structured_row_coverage(table: str, ask_obj: dict, rows: list[dict], row_flow: dict | None = None, selected_tickers: list | None = None) -> dict:
    row_flow = dict(row_flow or {})
    row_flow.setdefault("submitted_rows", len(rows))
    row_flow.setdefault("fed_rows", len(rows))
    is_universe_scope = str((ask_obj or {}).get("scope") or "").lower() == "universe"
    # scope="universe" answers count/list-all questions; a count row is not
    # expected to carry one row per requested ticker, so entity coverage is N/A.
    requested_tickers = [] if is_universe_scope else _unique_nonempty([str(t).upper() for t in ask_obj.get("tickers", [])])
    selected_tickers = [] if is_universe_scope else _unique_nonempty([str(t).upper() for t in (selected_tickers or [])])
    if requested_tickers and set(selected_tickers) == set(requested_tickers):
        selected_tickers = []
    expected_tickers = selected_tickers or requested_tickers
    fed_tickers = [] if is_universe_scope else _unique_nonempty([str(r.get("ticker") or "").upper() for r in rows])
    missing_tickers = [] if is_universe_scope else [t for t in expected_tickers if t not in fed_tickers]
    unselected_requested_tickers = [] if is_universe_scope else [t for t in requested_tickers if selected_tickers and t not in selected_tickers]
    periods = _unique_nonempty([r.get("period_end") or r.get("date") or r.get("year") for r in rows])
    columns = list(rows[0].keys()) if rows else []
    coverage = {
        "source_table": table,
        "row_flow": row_flow,
        "requested_tickers": requested_tickers,
        "selected_tickers": selected_tickers,
        "expected_tickers": expected_tickers,
        "fed_tickers": fed_tickers,
        "missing_tickers": missing_tickers,
        "unselected_requested_tickers": unselected_requested_tickers,
        "periods_present": periods,
        "columns_present": columns,
        "notes": [],
    }
    if row_flow.get("fed_rows") != row_flow.get("submitted_rows"):
        coverage["notes"].append(f"Partial table: fed {row_flow.get('fed_rows')} of {row_flow.get('submitted_rows')} submitted rows.")
    if is_universe_scope:
        coverage["notes"].append("Universe enumeration: ticker-level row coverage is not applicable.")
    if selected_tickers:
        coverage["notes"].append("Selector output narrowed the requested universe to: " + ", ".join(selected_tickers))
    if missing_tickers:
        coverage["notes"].append("Expected tickers absent from fed rows: " + ", ".join(missing_tickers))
    if table == "ratios":
        ratio_names_present = _unique_nonempty([r.get("ratio_name") for r in rows])
        requested_ratio_names = _unique_nonempty([str(r).lower() for r in ask_obj.get("ratio_names", []) if str(r).strip()])
        present_ratio_lower = {str(r).lower() for r in ratio_names_present}
        missing_ratio_names = [r for r in requested_ratio_names if r not in present_ratio_lower]
        coverage["ratio_names_present"] = ratio_names_present
        coverage["requested_ratio_names"] = requested_ratio_names
        coverage["missing_ratio_names"] = missing_ratio_names
        if missing_ratio_names:
            coverage["notes"].append("Requested ratio_name values absent from fed rows: " + ", ".join(missing_ratio_names))
        mixed = []
        by_ticker = {}
        for row in rows:
            ticker = str(row.get("ticker") or "").upper()
            period = str(row.get("period_end") or "").strip()
            if ticker and period:
                by_ticker.setdefault(ticker, []).append(period)
        for ticker, ticker_periods in by_ticker.items():
            unique_periods = _unique_nonempty(ticker_periods)
            if len(unique_periods) > 1:
                gap = _date_gap_days(unique_periods)
                mixed.append({"ticker": ticker, "periods": unique_periods, "max_gap_days": gap})
        if mixed:
            coverage["mixed_period_tickers"] = mixed
            coverage["notes"].append("Ratio rows include multiple period_end values within ticker; compare timing carefully.")
    if table == "financial_statements":
        coverage["canonical_names_present"] = _unique_nonempty([r.get("canonical_name") for r in rows])
        coverage["statement_types_present"] = _unique_nonempty([r.get("statement_type") for r in rows])
    if table == "market_data":
        dates = [_parse_iso_date(r.get("date")) for r in rows if r.get("date")]
        dates = [d for d in dates if d]
        if dates:
            max_date = max(dates)
            stale = []
            for row in rows:
                row_date = _parse_iso_date(row.get("date"))
                ticker = str(row.get("ticker") or "").upper()
                if row_date and ticker:
                    lag = (max_date - row_date).days
                    if lag > 30:
                        stale.append({"ticker": ticker, "date": str(row_date), "lag_days": lag})
            coverage["market_data_max_date"] = str(max_date)
            if stale:
                coverage["stale_market_rows"] = stale
                coverage["notes"].append("Some market_data rows lag the latest date in this attachment by more than 30 days.")
        null_fields = {}
        market_fields = ["market_cap", "pe_ratio", "pb_ratio", "dividend_yield", "enterprise_value"]
        for row in rows:
            ticker = str(row.get("ticker") or "").upper()
            missing = [field for field in market_fields if field in row and row.get(field) in (None, "")]
            if ticker and missing:
                null_fields[ticker] = sorted(set(missing))
        if null_fields:
            coverage["null_market_fields_by_ticker"] = null_fields
            coverage["notes"].append("Some market_data fields are null; treat them as unavailable, not zero.")
    return coverage


def _coverage_summary_text(coverage: dict) -> str:
    if not coverage:
        return ""
    flow = coverage.get("row_flow", {})
    parts = []
    if flow:
        submitted = flow.get("submitted_rows")
        fed = flow.get("fed_rows")
        final_sql = flow.get("final_sql_rows")
        if submitted is not None and fed is not None:
            text = f"rows fed={fed} of submitted={submitted}"
            if final_sql is not None:
                text += f"; final_sql_rows={final_sql}"
            parts.append(text)
    if coverage.get("requested_tickers"):
        parts.append("requested tickers=" + ", ".join(coverage.get("requested_tickers", [])))
    if coverage.get("selected_tickers"):
        parts.append("selected tickers=" + ", ".join(coverage.get("selected_tickers", [])))
    if coverage.get("fed_tickers"):
        parts.append("fed tickers=" + ", ".join(coverage.get("fed_tickers", [])))
    if coverage.get("missing_tickers"):
        parts.append("missing tickers=" + ", ".join(coverage.get("missing_tickers", [])))
    if coverage.get("ratio_names_present"):
        parts.append("ratios=" + ", ".join(coverage["ratio_names_present"][:12]))
    if coverage.get("missing_ratio_names"):
        parts.append("missing ratios=" + ", ".join(coverage["missing_ratio_names"][:12]))
    if coverage.get("canonical_names_present"):
        parts.append("canonicals=" + ", ".join(coverage["canonical_names_present"][:12]))
    if coverage.get("periods_present"):
        parts.append("periods=" + ", ".join(coverage["periods_present"][:8]))
    if coverage.get("market_data_max_date"):
        parts.append("market max date=" + coverage["market_data_max_date"])
    notes = coverage.get("notes") or []
    if notes:
        parts.append("notes=" + " | ".join(notes[:3]))
    return "; ".join(parts)

In [ ]:
# ── §2.9  Submit validation invariants ────────────────────────────────
#
#  INVARIANT LAYER — deterministic hard guards that run on every submit attempt,
#  regardless of what the LLM decided. These are code, not prompt engineering.
#    • is_discovery_sql()       — rejects bare SELECT DISTINCT / COUNT / SELECT *
#    • has_useful_structured_rows() — requires at least one numeric metric column
#    • validate_submit()        — enforces ticker coverage, no global MAX() trap,
#                                no unresolved tickers, balanced narrative selection
#  In:  table, ask_obj, final_sql, final_rows  |  Out: (errors list, rows) — raises on fail

def is_discovery_sql(sql: str) -> bool:
    lower = sql.lower().strip()
    if not lower:
        return True
    # Queries with joins or CTEs that join are analytical by construction — real
    # discovery queries are flat single-table selects. Skip the DISTINCT heuristic
    # for joined queries so CTEs like `WITH ranked AS (SELECT DISTINCT ...) SELECT ... JOIN ranked ...`
    # are not false-flagged as discovery.
    has_join = bool(re.search(r"\bjoin\b", lower))
    if not has_join:
        # Only flag if the OUTERMOST select is a pure discovery select.
        # Use re.match (anchored to start) so subqueries like
        # `... AND period_end IN (SELECT DISTINCT period_end ...)` are NOT caught.
        if re.match(r"\s*select\s+distinct\s+(canonical_name|period_end|ticker|year|filing_period)\b", lower):
            return True
    if re.search(r"\bselect\b[\s\S]*\bcount\s*\(", lower) and re.search(r"\bgroup\s+by\b", lower) and not has_join:
        return True
    if re.match(r"\s*select\s+\*\s+from\b", lower):
        return True
    return False


def _is_enumeration_ask(ask_obj: dict, final_sql: str = "") -> bool:
    # scope="universe" = count/list-all asks; relax discovery and value-column guards.
    # scope="market_wide" = ranking/screen asks; still require analytical metric rows.
    return str(ask_obj.get("scope") or "").lower() == "universe"


def has_useful_structured_rows(rows: list, allow_enumeration: bool = False) -> bool:
    # Contract: structured submit must have rows, and must include at least one
    # value/metric column with a numeric value — unless this is a scope="universe"
    # enumeration ask (count/list), in which case non-empty rows are sufficient.
    if not rows:
        return False
    if allow_enumeration:
        # Universe/enumeration asks (count, list-all): non-empty rows are the contract.
        return True
    # Non-enumeration: require at least one numeric value in the result set.
    # This rejects pure metadata dumps (ticker, period_end, canonical_name only)
    # while accepting any result with a measurement column.
    return any(
        isinstance(v, (int, float)) and not isinstance(v, bool)
        for row in rows for v in row.values()
    )


def narrative_selection_limit(ask_obj: dict) -> int:
    ask_tickers = [str(t).upper() for t in ask_obj.get("tickers", []) if str(t).strip()]
    return max(MAX_SELECTED_SOURCE_IDS, len(dict.fromkeys(ask_tickers)))


def select_balanced_narrative_ids(candidates: list, ask_obj: dict, max_ids: int | None = None) -> list[int]:
    if not candidates:
        return []
    ask_tickers = list(dict.fromkeys(str(t).upper() for t in ask_obj.get("tickers", []) if str(t).strip()))
    limit = max_ids or narrative_selection_limit(ask_obj)
    selected, seen = [], set()

    for ticker in ask_tickers:
        bucket = [c for c in candidates if str(c.get("ticker") or "").upper() == ticker]
        for c in rank_candidates(bucket)[:1]:
            if c["id"] in seen:
                continue
            selected.append(c["id"])
            seen.add(c["id"])
            break

    for c in rank_candidates(candidates):
        if len(selected) >= limit:
            break
        if c["id"] in seen:
            continue
        selected.append(c["id"])
        seen.add(c["id"])
    return selected



def validate_submit(table: str, ask_obj: dict, final_sql: str, final_rows: list, current_candidates: list, source_ids: list) -> tuple[list, list]:
    errors = []
    unresolved = [str(t).upper() for t in ask_obj.get("_unresolved_tickers", []) if str(t).strip()]
    if final_sql and unresolved:
        sql_upper = final_sql.upper()
        leaked = [t for t in unresolved if re.search(rf"(?<![A-Z0-9_]){re.escape(t)}(?![A-Z0-9_])", sql_upper)]
        if leaked:
            errors.append(
                "SQL references unresolved/out-of-coverage ticker(s): "
                + ", ".join(leaked)
                + ". Use only resolved tickers from ask_object['tickers']; unresolved tickers are diagnostics for analyst disclosure, not retrieval scope."
            )
            return errors, []
    if table != "financial_documents_new":
        if not final_sql.strip():
            errors.append("final_sql is empty")
            return errors, []
        allow_enumeration = _is_enumeration_ask(ask_obj, final_sql)
        if is_discovery_sql(final_sql) and not allow_enumeration:
            errors.append("final SQL is discovery-only")
            return errors, []
        # Reject global latest-period subqueries for multi-ticker asks.
        # These silently drop tickers whose latest period_end differs from the global max.
        _ask_tickers_check = ask_obj.get("tickers", [])
        if len(_ask_tickers_check) >= 2:
            _sql_lower = final_sql.lower()
            _has_global_max = bool(re.search(
                r"(?:\b\w+\.)?(?:period_end|date)\s*(?:in\s*\(|=\s*\()\s*select\s+max\s*\(\s*(?:\b\w+\.)?(?:period_end|date)\s*\)",
                _sql_lower,
            ))
            if _has_global_max:
                errors.append(
                    "Multi-ticker SQL uses a global MAX(period_end/date) subquery — IN or = form — "
                    "which silently drops tickers whose latest period_end differs from the global max. "
                    "Use ROW_NUMBER() OVER (PARTITION BY ticker, canonical_name ORDER BY period_end DESC) "
                    "and filter rn <= N to get correct latest-per-ticker rows."
                )
                return errors, []
        # Re-execute final_sql so the row check reflects what was submitted, not whatever run_sql happened to run last.
        exec_result = run_sql(final_sql)
        rows_to_check = exec_result.get("rows", []) if exec_result.get("success") else final_rows
        if not has_useful_structured_rows(rows_to_check, allow_enumeration=allow_enumeration):
            errors.append("result rows have no analytical columns")
        # Multi-ticker coverage: for named compare/trend asks (small ticker set,
        # not router-marked as selector/ranking), every requested ticker must
        # appear in the submitted rows.
        _ask_tickers_cov = [str(t).upper() for t in ask_obj.get("tickers", []) if str(t).strip()]
        _is_small_compare = 2 <= len(_ask_tickers_cov) <= 6
        _is_selector_ask = bool(ask_obj.get("is_selector"))
        if _is_small_compare and not _is_selector_ask and rows_to_check:
            _tickers_in_rows = {str(r.get("ticker", "")).upper() for r in rows_to_check}
            _missing_cov = [t for t in _ask_tickers_cov if t not in _tickers_in_rows]
            if _missing_cov:
                errors.append(
                    f"Multi-ticker coverage gap: submitted rows missing data for "
                    f"{', '.join(_missing_cov)}. Re-query using "
                    "a CTE/subquery with ROW_NUMBER() OVER (PARTITION BY ticker, metric ORDER BY period_end DESC) AS rn, "
                    "then filter WHERE rn <= N so every requested ticker is represented."
                )
        if table == "ratios" and ask_obj.get("ratio_names") and rows_to_check:
            _present_ratios = {str(r.get("ratio_name", "")).lower() for r in rows_to_check if str(r.get("ratio_name", "")).strip()}
            _missing_ratios = [str(r).lower() for r in ask_obj.get("ratio_names", []) if str(r).strip() and str(r).lower() not in _present_ratios]
            if _missing_ratios:
                errors.append(
                    "Ratio coverage gap: submitted rows missing requested ratio_name(s): "
                    + ", ".join(_missing_ratios)
                    + ". Re-query with WHERE ratio_name IN (...) for all requested ratios, or submit an explicit insufficiency result that discloses unavailable metrics."
                )
        return errors, rows_to_check

    if source_ids:
        selected_ids = set(source_ids)
    else:
        selected_ids = set(select_balanced_narrative_ids(current_candidates, ask_obj))

    if not selected_ids:
        errors.append("no narrative source candidates available")
        return errors, []

    ask_tickers = list(dict.fromkeys(str(t).upper() for t in ask_obj.get("tickers", []) if str(t).strip()))
    # Selector asks scan a broad sector to find winners — coverage across every
    # ticker in scope is not required (the analyst picks the top-N from what was found).
    _is_selector = bool(ask_obj.get("is_selector"))
    if len(ask_tickers) >= 2 and not _is_selector:
        covered = {str(c["ticker"]).upper() for c in current_candidates if c["id"] in selected_ids}
        missing = [ticker for ticker in ask_tickers if ticker not in covered]
        if missing:
            errors.append(
                f"missing pages for ticker(s): {', '.join(missing)}. "
                "Use UNION ALL per-ticker subqueries (one subquery per ticker, each with its own LIMIT) "
                "to guarantee balanced retrieval — a global ticker IN (...) LIMIT N will omit low-scoring tickers. "
                f"For each missing ticker, add a subquery with broader MATCH terms drawn from concepts[]. "
                "If no pages exist after broadening, submit with covered tickers and note the gap in the title."
            )
    return errors, []

In [ ]:
# ── §2.10a  Provenance parse helpers ────────────────────────────────────────────
#
#  Structured rows carry source information in columns like source_url and source_pages.
#  These helpers parse those raw column values into (ticker, filing_year, page_number)
#  primitives that the recovery functions in §2.10b can use to query D1.
#  In:  raw column values (strings, JSON-encoded lists, URL paths)
#  Out: parsed primitives — page number lists, filing years, ticker strings

def _parse_source_pages_value(value) -> list:
    if isinstance(value, str):
        try:
            value = json.loads(value)
        except Exception:
            value = [int(p.strip()) for p in value.replace('[', '').replace(']', '').split(',') if p.strip().isdigit()]
    if not isinstance(value, list):
        return []
    return [int(x) for x in value if isinstance(x, (int, float)) and int(x) > 0]


def _add_lookup_pages(page_map: dict, ticker: str, filing_key: str, pages: list):
    clean_ticker = str(ticker or '').strip().upper()
    clean_key = str(filing_key or '').strip()
    if not clean_ticker or not clean_key or not pages:
        return
    bucket = page_map.setdefault((clean_ticker, clean_key), set())
    for page in pages:
        if isinstance(page, (int, float)) and int(page) > 0:
            bucket.add(int(page))


def _parse_filing_year_from_source_url(source_url) -> str | None:
    text = str(source_url or '').strip()
    if not text:
        return None
    m = re.search(r'/(\d{4})/[^/]*$', text)
    return m.group(1) if m else None


def _parse_ticker_from_source_url(source_url) -> str | None:
    text = str(source_url or '').strip()
    if not text:
        return None
    m = re.search(r'/PDF_PAGES/([^/]+)/', text, re.I)
    return m.group(1).upper() if m else None


def _parse_management_source_pdf(source_pdf) -> tuple[str, str, int] | None:
    text = str(source_pdf or '').strip()
    if not text:
        return None
    m = re.search(r'^([^/]+)/(\d{4})/.*/page_(\d+)\.pdf$', text, re.I)
    if not m:
        return None
    return m.group(1).upper(), m.group(2), int(m.group(3))


def _resolve_page_map_to_doc_ids(page_map: dict) -> list:
    out = []
    for (ticker, filing_key), pages in page_map.items():
        unique_pages = sorted(pages)
        if not unique_pages:
            continue
        ph = ','.join(['?'] * len(unique_pages))
        if str(filing_key).startswith('YEAR:'):
            sql = f"SELECT id FROM financial_documents_new WHERE ticker = ? AND filing_year = ? AND pg IN ({ph})"
            binds = [ticker, filing_key.replace('YEAR:', ''), *unique_pages]
        else:
            sql = f"SELECT id FROM financial_documents_new WHERE ticker = ? AND filing_period = ? AND pg IN ({ph})"
            binds = [ticker, filing_key, *unique_pages]
        res = run_sql(sql, binds)
        for row in res.get('rows', []):
            pid = row.get('id')
            if isinstance(pid, int):
                out.append(pid)
    return list(dict.fromkeys(out))


RATIO_PROVENANCE_MAP = {
    'roe': [('profit_loss', 'net_profit', 'same'), ('balance_sheet', 'equity', 'pit')],
    'roa': [('profit_loss', 'net_profit', 'same'), ('balance_sheet', 'total_assets', 'pit')],
    'net_margin': [('profit_loss', 'net_profit', 'same'), ('profit_loss', 'revenue_net', 'same')],
    'gross_margin': [('profit_loss', 'gross_profit', 'same'), ('profit_loss', 'revenue_net', 'same')],
    'current_ratio': [('balance_sheet', 'current_assets', 'pit'), ('balance_sheet', 'current_liabilities', 'pit')],
    'debt_to_equity': [('balance_sheet', 'total_liabilities', 'pit'), ('balance_sheet', 'equity', 'pit')],
    'asset_turnover': [('profit_loss', 'revenue_net', 'same'), ('balance_sheet', 'total_assets', 'pit')],
}

print("Provenance parse helpers ready.")

In [ ]:
# ── §2.10b  Provenance recovery functions ────────────────────────────────────────
#
#  After lanes complete, we need to recover which source PAGES each structured row
#  came from — so the analyst can cite them and the source panel can display them.
#  Done deterministically (no LLM): walk rows → extract source identifiers (via §2.10a)
#  → batch-query D1 for matching document IDs → attach provenanceIds to each artifact.
#  In:  table name + rows list (from structured lane artifact)
#  Out: provenanceIds (list[int]) — D1 document IDs for source-panel display

def _recover_financial_statements_provenance(rows: list) -> list:
    page_map = {}
    lookup_keys = {}
    for row in rows:
        ticker = str(row.get('ticker') or _parse_ticker_from_source_url(row.get('source_url')) or '').strip().upper()
        period_end = str(row.get('period_end') or row.get('filing_period') or '').strip()
        pages = _parse_source_pages_value(row.get('source_pages'))
        _add_lookup_pages(page_map, ticker, period_end, pages)
        if not ticker or not period_end:
            continue
        canonical_name = str(row.get('canonical_name') or '').strip()
        statement_type = str(row.get('statement_type') or '').strip()
        section = str(row.get('section') or '').strip()
        original_name = str(row.get('original_name') or '').strip()
        key = (ticker, period_end, canonical_name, statement_type, section, original_name)
        lookup_keys[key] = {
            'ticker': ticker,
            'period_end': period_end,
            'canonical_name': canonical_name,
            'statement_type': statement_type,
            'section': section,
            'original_name': original_name,
        }
    for key in lookup_keys.values():
        conditions = ['ticker = ?', 'period_end = ?', 'source_pages IS NOT NULL']
        binds = [key['ticker'], key['period_end']]
        if key['canonical_name']:
            conditions.append('canonical_name = ?')
            binds.append(key['canonical_name'])
        if key['statement_type']:
            conditions.append('statement_type = ?')
            binds.append(key['statement_type'])
        if key['section']:
            conditions.append('section = ?')
            binds.append(key['section'])
        if key['original_name']:
            conditions.append('original_name = ?')
            binds.append(key['original_name'])
        sql = 'SELECT ticker, period_end, source_pages, source_url FROM financial_statements WHERE ' + ' AND '.join(conditions)
        res = run_sql(sql, binds)
        for row in res.get('rows', []):
            ticker = str(row.get('ticker') or '').strip().upper()
            period_end = str(row.get('period_end') or '').strip()
            pages = _parse_source_pages_value(row.get('source_pages'))
            _add_lookup_pages(page_map, ticker, period_end, pages)
    return _resolve_page_map_to_doc_ids(page_map)


def _recover_ratios_provenance(rows: list) -> list:
    page_map = {}
    # Batch-rehydrate rows missing section or period_duration.
    # Ratios SQL may not include these columns; recover them from the ratios table
    # so component lookups use the correct period_duration (12M vs LTM vs PIT).
    _missing_meta = [
        (str(r.get('ticker') or '').strip().upper(),
         str(r.get('period_end') or '').strip(),
         str(r.get('ratio_name') or '').strip().lower())
        for r in rows
        if (not r.get('section') or not r.get('period_duration'))
           and str(r.get('ticker') or '').strip()
           and str(r.get('period_end') or '').strip()
           and str(r.get('ratio_name') or '').strip()
    ]
    _meta_lookup: dict[tuple, dict] = {}
    _reh_grouped: dict[tuple, list] = {}
    if _missing_meta:
        _tickers = list(dict.fromkeys(t for t, _, _ in _missing_meta))
        _periods = list(dict.fromkeys(p for _, p, _ in _missing_meta))
        _names = list(dict.fromkeys(n for _, _, n in _missing_meta))
        _ph_t = ','.join(['?'] * len(_tickers))
        _ph_p = ','.join(['?'] * len(_periods))
        _ph_n = ','.join(['?'] * len(_names))
        _reh_sql = (
            f"SELECT ticker, period_end, ratio_name, section, period_duration "
            f"FROM ratios "
            f"WHERE ticker IN ({_ph_t}) AND period_end IN ({_ph_p}) AND ratio_name IN ({_ph_n})"
        )
        _reh_res = run_sql(_reh_sql, [*_tickers, *_periods, *_names])
        for _r in _reh_res.get('rows', []):
            _key = (
                str(_r.get('ticker') or '').strip().upper(),
                str(_r.get('period_end') or '').strip(),
                str(_r.get('ratio_name') or '').strip().lower(),
            )
            _reh_grouped.setdefault(_key, []).append({
                'section': str(_r.get('section') or '').strip(),
                'period_duration': str(_r.get('period_duration') or '').strip().upper(),
            })
        # Pick the best candidate per key: prefer consolidated, then ratio-family-aware duration.
        # Prevents last-row-wins ambiguity when both 12M and LTM rows exist for the same triple.
        _BS_RATIOS = frozenset({'current_ratio', 'debt_to_equity'})
        _MARGIN_RATIOS = frozenset({'gross_margin', 'net_margin'})

        def _duration_rank_for_ratio(ratio_name: str, ticker: str) -> dict:
            # Balance-sheet ratios: PIT is always correct.
            if ratio_name in _BS_RATIOS:
                return {'PIT': 0, 'LTM': 1, '12M': 2, '3M': 3}
            # Margin ratios: LTM for banks, 12M for non-banks.
            if ratio_name in _MARGIN_RATIOS:
                _ind = (globals().get('TICKER_INDUSTRY_MAP') or {}).get(ticker, '')
                if 'bank' in str(_ind).lower():
                    return {'LTM': 0, '12M': 1, '3M': 2, 'PIT': 3}
            # Default annual flow ratios (roe, roa, asset_turnover, non-bank margins):
            # 12M preferred; LTM accepted as equivalent for annual.
            return {'12M': 0, 'LTM': 1, '3M': 2, 'PIT': 3}

        for _key, _candidates in _reh_grouped.items():
            _t, _pe, _rn = _key
            _rank = _duration_rank_for_ratio(_rn, _t)
            _meta_lookup[_key] = min(
                _candidates,
                key=lambda m, _r=_rank: (
                    0 if m.get('section') == 'consolidated' else 1,
                    _r.get(m.get('period_duration', ''), 4),
                ),
            )

    for row in rows:
        ticker = str(row.get('ticker') or '').strip().upper()
        period_end = str(row.get('period_end') or '').strip()
        ratio_name = str(row.get('ratio_name') or '').strip().lower()
        # Use rehydrated metadata if the row itself was missing section/period_duration.
        _rehydrated = _meta_lookup.get((ticker, period_end, ratio_name), {})
        # Default to consolidated — ratio queries almost always use this section.
        section = str(row.get('section') or _rehydrated.get('section') or 'consolidated').strip()
        period_duration = str(row.get('period_duration') or _rehydrated.get('period_duration') or '').strip().upper()
        if not ticker or not period_end or not ratio_name:
            continue
        for statement_type, canonical_name, duration_mode in RATIO_PROVENANCE_MAP.get(ratio_name, []):
            preferred_duration = None
            if duration_mode == 'same' and period_duration:
                preferred_duration = '12M' if period_duration == 'LTM' else period_duration
            elif duration_mode == 'pit':
                preferred_duration = 'PIT'
            attempts = [preferred_duration] if preferred_duration else []
            attempts.append(None)
            for duration in attempts:
                conditions = ['ticker = ?', 'period_end = ?', 'canonical_name = ?', 'statement_type = ?', 'source_pages IS NOT NULL', 'section = ?']
                binds = [ticker, period_end, canonical_name, statement_type, section]
                if duration:
                    conditions.append('period_duration = ?')
                    binds.append(duration)
                sql = 'SELECT ticker, period_end, source_pages FROM financial_statements WHERE ' + ' AND '.join(conditions)
                res = run_sql(sql, binds)
                if not res.get('rows'):
                    continue
                for match in res.get('rows', []):
                    pages = _parse_source_pages_value(match.get('source_pages'))
                    # Use exact period_end only — no YEAR fallback, which fans out
                    # across all quarterly/annual filings sharing the same page numbers.
                    _add_lookup_pages(page_map, match.get('ticker'), match.get('period_end'), pages)
                break
    return _resolve_page_map_to_doc_ids(page_map)


def _recover_management_comp_provenance(rows: list) -> list:
    page_map = {}
    for row in rows:
        direct = _parse_management_source_pdf(row.get('source_pdf') or row.get('source_url'))
        if direct:
            ticker, year, page = direct
            _add_lookup_pages(page_map, ticker, f'YEAR:{year}', [page])
            continue
        ticker = str(row.get('ticker') or '').strip().upper()
        year = str(row.get('year') or row.get('filing_year') or '').strip()
        section = str(row.get('section') or row.get('reporting_scope') or '').strip()
        role = str(row.get('role') or '').strip()
        if not ticker or not year:
            continue
        conditions = ['ticker = ?', 'year = ?', 'source_pdf IS NOT NULL']
        binds = [ticker, year]
        if section:
            conditions.append('section = ?')
            binds.append(section)
        if role:
            conditions.append('role = ?')
            binds.append(role)
        sql = 'SELECT source_pdf FROM management_comp WHERE ' + ' AND '.join(conditions)
        res = run_sql(sql, binds)
        for match in res.get('rows', []):
            parsed = _parse_management_source_pdf(match.get('source_pdf'))
            if parsed:
                pticker, pyear, page = parsed
                _add_lookup_pages(page_map, pticker, f'YEAR:{pyear}', [page])
    return _resolve_page_map_to_doc_ids(page_map)


def recover_structured_provenance(table: str, rows: list) -> list:
    if not rows:
        return []
    if table == 'financial_statements':
        return _recover_financial_statements_provenance(rows)
    if table == 'ratios':
        return _recover_ratios_provenance(rows)
    if table == 'management_comp':
        return _recover_management_comp_provenance(rows)
    return []


print("Provenance recovery functions ready.")

In [ ]:
# ── §2.11 Lane loop and narrative extraction helpers ─────────────────
#
#  The lane runs as an agentic loop (up to MAX_LANE_ITERATIONS turns).
#  These helpers manage the loop's message history and extraction lifecycle:
#    append_tool_call()           — adds LLM tool call + result to message history
#    should_auto_extract_narrative() — decides if post-submit extraction should fire
#    ensure_narrative_extractions() — runs extraction if lane submitted IDs without it
#  In:  messages list, artifact dict, ask_obj  |  Out: updated messages / extractions

def append_tool_call(messages: list, msg):
    messages.append({
        "role": "assistant",
        "content": msg.content or "",
        "tool_calls": [
            {
                "id": tc.id,
                "type": "function",
                "function": {"name": tc.function.name, "arguments": tc.function.arguments},
            }
            for tc in msg.tool_calls
        ],
    })


def should_auto_extract_narrative(ask_obj: dict, selected_ids: list[int]) -> bool:
    # Trigger on posture and ticker count only — no prose keyword parsing.
    # Extraction is cheap context compression; extract whenever pages were selected
    # and the ask is structured enough to benefit (numeric posture always, narrative
    # with multiple tickers always, single-ticker narrative always too since the cost
    # is low and the analyst benefits from structured excerpts over raw page text).
    if not selected_ids:
        return False
    posture = str(ask_obj.get("posture") or "").lower()
    if posture in {"numeric", "narrative"}:
        return True
    # Fallback: multi-ticker asks benefit from structured excerpts regardless of posture
    return len(ask_obj.get("tickers", []) or []) >= 2


def build_auto_extraction_prompt(ask_obj: dict, ask_text: str) -> str:
    posture = str(ask_obj.get("posture") or "").lower()
    concepts = [str(c) for c in ask_obj.get("concepts", []) if str(c).strip()]
    lines = [
        f"Task: {ask_text}",
        "Extract only evidence from this page that helps answer the task.",
        "Start with the ticker and period if visible.",
    ]
    if posture == "numeric":
        lines.extend([
            "For numeric evidence, extract the exact stated figures, labels, units, and period.",
            "Do not calculate ratios unless the ratio is explicitly stated on the page.",
            "If components are present, list them separately rather than computing a final value.",
        ])
    else:
        lines.extend([
            "For narrative evidence, extract the specific management/director commentary, not boilerplate.",
            "Prefer concise cause/effect, outlook, strategy, risk, and demand statements.",
        ])
    if concepts:
        lines.append("Target concepts: " + ", ".join(concepts[:12]))
    lines.append(EXTRACTION_RETURN_SPEC)
    return "\n".join(lines)


def ensure_narrative_extractions(artifact: dict, ask_obj: dict, ask_text: str, selected_ids: list[int]) -> None:
    if artifact.get("narrativeExtractions"):
        return
    if not should_auto_extract_narrative(ask_obj, selected_ids):
        return
    prompt = build_auto_extraction_prompt(ask_obj, ask_text)
    print(f"  auto extract_from_pages: {len(selected_ids)} pages")
    artifact["narrativeExtractions"] = extract_from_pages(selected_ids, prompt)


In [ ]:
# ── §2.12 Structured lane runner ─────────────────────────────────────
#
#  The structured lane is an LLM agent with exactly two tools: run_sql and submit.
#  It iterates up to MAX_LANE_ITERATIONS turns: each turn it either refines its SQL
#  or submits its final rows. validate_submit() (§2.9) guards every submission attempt.
#  Output carries both the data (analysisAttachment) and quality metadata (coverage).
#  In:  ask_obj + variable name (e.g. 'q1')
#  Out: artifact {analysisAttachment, finalQueries, coverage, selected_tickers, …}

def run_structured_lane(ask_obj: dict, variable: str = "q1") -> dict:
    table = ask_obj["table"]
    ask_text = ask_obj["ask"]
    system = build_lane_system(table, ask_obj)
    messages = [{"role": "system", "content": system}, {"role": "user", "content": ask_text}]
    artifact = {
        "variable": variable,
        "table": table,
        "ask": ask_text,
        "ask_object": ask_obj,
        "finalQueries": [],
        "resultSummary": "",
        "analysisAttachment": None,
        "provenanceIds": [],
        "narrativeIds": [],
        "narrativeExtractions": [],
        "selected_tickers": [],
        "_extraction_attempted": False,
    }
    last_rows, last_sql = [], ""
    sql_queries = []
    input_tokens = output_tokens = 0

    for iteration in range(MAX_LANE_ITERATIONS):
        print(f"\n[{variable}] {table} — iteration {iteration + 1}")
        if iteration == MAX_LANE_ITERATIONS - 1:
            messages.append({"role": "user", "content": "Final iteration. Your ONLY valid tool on this turn is `submit`. Do not call `run_sql` or any other tool — any non-submit tool call will be ignored and the lane will fail. Submit the current analytical rows if they answer the ask, or fail out."})

        response = client.chat.completions.create(
            model=MODEL_LANE,
            messages=messages,
            tools=[TOOL_RUN_SQL, TOOL_SUBMIT],
            tool_choice="auto",
        )
        input_tokens += response.usage.prompt_tokens
        output_tokens += response.usage.completion_tokens
        msg = response.choices[0].message

        if not msg.tool_calls:
            print("  No tool calls from model")
            messages.append({"role": "assistant", "content": msg.content or ""})
            if iteration < MAX_LANE_ITERATIONS - 1:
                messages.append({"role": "user", "content": "Use tools only: run_sql, inspect, then submit."})
            continue

        append_tool_call(messages, msg)

        for tc in msg.tool_calls:
            name = tc.function.name
            args = safe_json(tc.function.arguments, {})

            if name == "run_sql":
                sql = str(args.get("sql", "")).strip()
                print(f"  run_sql: {sql[:140]}{'...' if len(sql) > 140 else ''}")
                result = run_sql(sql)
                if not result["success"]:
                    messages.append({"role": "tool", "tool_call_id": tc.id, "content": json.dumps({"error": result["error"]})})
                    continue
                last_rows = result["rows"]
                last_sql = sql
                sql_queries.append(sql)
                if table in {"financial_statements", "ratios", "management_comp"} and result["rows"]:
                    artifact["provenanceIds"] = list(dict.fromkeys(artifact["provenanceIds"] + recover_structured_provenance(table, result["rows"])))
                print(f"    → {result['row_count']} rows")
                sample = [
                    {k: (str(v)[:200] if isinstance(v, str) else v) for k, v in list(row.items())[:15]}
                    for row in result["rows"][:MAX_SQL_RESULT_SAMPLE_ROWS]
                ]
                messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": json.dumps({
                        "row_count": result["row_count"],
                        "columns": result["columns"],
                        "sample": sample,
                        "truncated": result["row_count"] > MAX_SQL_RESULT_SAMPLE_ROWS,
                    }),
                })
                continue

            if name == "submit":
                title = str(args.get("title", f"{table} result")).strip()
                final_sql = str(args.get("final_sql", last_sql)).strip()
                selected_tickers = [str(t).strip().upper() for t in (args.get("selected_tickers") or []) if str(t).strip()]
                errors, submit_rows = validate_submit(table, ask_obj, final_sql, last_rows, [], [])
                if errors:
                    messages.append({
                        "role": "tool",
                        "tool_call_id": tc.id,
                        "content": json.dumps({
                            "error": f"Submission rejected: {'; '.join(errors)}",
                            "required_next_step": "Run analytical SQL that directly answers the ask, then submit again.",
                        }),
                    })
                    continue
                attachment_rows = submit_rows if submit_rows else last_rows
                artifact["finalQueries"] = [final_sql]
                row_flow = {"last_run_rows": len(last_rows), "final_sql_rows": len(submit_rows), "submitted_rows": len(attachment_rows), "fed_rows": len(attachment_rows)}
                artifact["analysisAttachment"] = _build_table_attachment(variable, title, attachment_rows, table=table, ask_obj=ask_obj, row_flow=row_flow, selected_tickers=selected_tickers)
                artifact["coverage"] = artifact["analysisAttachment"].get("coverage", {})
                artifact["selected_tickers"] = artifact["coverage"].get("selected_tickers", [])
                artifact["resultSummary"] = f"{title}: {len(attachment_rows)} rows"
                messages.append({"role": "tool", "tool_call_id": tc.id, "content": json.dumps({"ok": True, "title": title})})
                artifact["_debug"] = {
                    "sql_queries": sql_queries,
                    "message_history": messages,
                    "system_prompt": system,
                    "input_tokens": input_tokens,
                    "output_tokens": output_tokens,
                }
                return artifact

            messages.append({"role": "tool", "tool_call_id": tc.id, "content": json.dumps({"error": f"Unknown tool: {name}"})})

        if iteration < MAX_LANE_ITERATIONS - 1:
            messages.append({"role": "user", "content": "Run one bounded next step only, then submit when valid."})

    artifact["resultSummary"] = f"lane_failed_no_valid_submit_after_{MAX_LANE_ITERATIONS}_iterations"
    artifact["_debug"] = {
        "sql_queries": sql_queries,
        "message_history": messages,
        "system_prompt": system,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
    }
    return artifact


In [ ]:
# ── §2.13 Narrative lane runner ──────────────────────────────────────
#
#  The narrative lane has four tool types: run_sql, submit, shaping, extract_from_pages.
#  It runs BM25 queries against financial_documents_new, shapes the candidate set,
#  extracts structured evidence from the best pages, and submits page IDs.
#  Auto-extraction fires post-submit if the lane submitted IDs without extracting.
#  In:  ask_obj + variable name
#  Out: artifact {narrativeIds, narrativeExtractions, selected_tickers, …}

def run_narrative_lane(ask_obj: dict, variable: str = "q1") -> dict:
    table = "financial_documents_new"
    ask_text = ask_obj["ask"]
    system = build_lane_system(table, ask_obj)
    tools = [TOOL_RUN_SQL, TOOL_SUBMIT] + SHAPING_TOOLS + [TOOL_EXTRACT_FROM_PAGES]
    messages = [{"role": "system", "content": system}, {"role": "user", "content": ask_text}]
    artifact = {
        "variable": variable,
        "table": table,
        "ask": ask_text,
        "ask_object": ask_obj,
        "finalQueries": [],
        "resultSummary": "",
        "analysisAttachment": None,
        "provenanceIds": [],
        "narrativeIds": [],
        "narrativeExtractions": [],
        "selected_tickers": [],
        "_extraction_attempted": False,
    }
    current_candidates = []
    inventory = {}  # pages kept by model via keep_ids across SQL calls
    candidate_history = []
    last_rows, last_sql = [], ""
    sql_queries = []
    input_tokens = output_tokens = 0

    for iteration in range(MAX_LANE_ITERATIONS):
        print(f"\n[{variable}] {table} — iteration {iteration + 1}")
        if iteration == MAX_LANE_ITERATIONS - 1:
            messages.append({"role": "user", "content": "Final iteration. Your ONLY valid tool on this turn is `submit`. Do not call `run_sql` or any shaping tool — any non-submit tool call will be ignored and the lane will fail. Submit the current candidate set if it covers every ticker in the ask, or fail out."})

        response = client.chat.completions.create(
            model=MODEL_LANE,
            messages=messages,
            tools=tools,
            tool_choice="auto",
        )
        input_tokens += response.usage.prompt_tokens
        output_tokens += response.usage.completion_tokens
        msg = response.choices[0].message

        if not msg.tool_calls:
            print("  No tool calls from model")
            messages.append({"role": "assistant", "content": msg.content or ""})
            if iteration < MAX_LANE_ITERATIONS - 1:
                messages.append({"role": "user", "content": "Use tools only: retrieve pages, shape them if needed, optionally extract, then submit."})
            continue

        append_tool_call(messages, msg)

        for tc in msg.tool_calls:
            name = tc.function.name
            args = safe_json(tc.function.arguments, {})

            if name == "run_sql":
                sql = str(args.get("sql", "")).strip()
                print(f"  run_sql: {sql[:140]}{'...' if len(sql) > 140 else ''}")
                result = run_sql(sql)
                if not result["success"]:
                    messages.append({"role": "tool", "tool_call_id": tc.id, "content": json.dumps({"error": result["error"]})})
                    continue
                last_rows = result["rows"]
                last_sql = sql
                sql_queries.append(sql)
                print(f"    → {result['row_count']} rows")

                new_candidates = normalize_narrative_rows(result["rows"])
                ask_tickers = [str(t).upper() for t in ask_obj.get("tickers", []) if str(t).strip()]
                # Honour keep_ids: move selected pages from current set into inventory
                for kid in [int(i) for i in args.get("keep_ids", []) if str(i).isdigit()]:
                    match = next((c for c in current_candidates if c["id"] == kid), None)
                    if match:
                        inventory[kid] = match
                # Merge inventory with new retrieval; inventory pages survive SQL replacement
                merged = {c["id"]: c for c in inventory.values()}
                for c in new_candidates:
                    merged[c["id"]] = c
                if merged:
                    current_candidates = rank_candidates(list(merged.values()))
                    candidate_history.append({
                        "step": f"retrieval_{len(candidate_history) + 1}",
                        "sql": sql,
                        "candidate_ids": [c["id"] for c in current_candidates],
                    })

                sample = [
                    {k: (str(v)[:200] if isinstance(v, str) else v) for k, v in list(row.items())[:15]}
                    for row in result["rows"][:MAX_SQL_RESULT_SAMPLE_ROWS]
                ]
                messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": json.dumps({
                        "row_count": result["row_count"],
                        "columns": result["columns"],
                        "sample": sample,
                        "candidate_count": len(current_candidates),
                        "candidate_summary": summarize_candidates(current_candidates),
                    }),
                })
                continue

            if name in SHAPING_TOOL_NAMES:
                print(f"  shaping: {name}")
                current_candidates = execute_shaping_tool(name, args, current_candidates)
                candidate_history.append({
                    "step": name,
                    "candidate_ids": [c["id"] for c in current_candidates],
                })
                messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": json.dumps({
                        "tool": name,
                        "candidates_remaining": len(current_candidates),
                        "candidate_summary": summarize_candidates(current_candidates),
                    }),
                })
                continue

            if name == "extract_from_pages":
                requested = [int(i) for i in args.get("page_ids", []) if int(i) > 0]
                active_ids = {c["id"] for c in current_candidates}
                scoped_ids = [i for i in requested if i in active_ids] if requested else [c["id"] for c in rank_candidates(current_candidates)]
                prompt = str(args.get("extraction_prompt") or "").strip() or build_default_extraction_prompt(ask_text)
                print(f"  extract_from_pages: {len(scoped_ids)} pages")
                extractions = extract_from_pages(scoped_ids, prompt)
                artifact["narrativeExtractions"] = extractions
                artifact["_extraction_attempted"] = True
                messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": json.dumps({
                        "extraction_count": len(extractions),
                        "extracted_page_ids": [e["page_id"] for e in extractions[:12]],
                    }),
                })
                continue

            if name == "submit":
                title = str(args.get("title", f"{table} result")).strip()
                final_sql = str(args.get("final_sql", last_sql)).strip()
                source_ids = [int(i) for i in (args.get("source_ids") or []) if int(i) > 0]
                selected_tickers = [str(t).strip().upper() for t in (args.get("selected_tickers") or []) if str(t).strip()]

                # Let the model's source_ids stand. validate_submit enforces
                # per-ticker coverage and rejects if inadequate, triggering a
                # requery. Auto-select only when the model submitted no source_ids
                # at all (handled inside validate_submit's fallback path).
                errors, _ = validate_submit(table, ask_obj, final_sql, last_rows, current_candidates, source_ids)
                if errors:
                    messages.append({
                        "role": "tool",
                        "tool_call_id": tc.id,
                        "content": json.dumps({
                            "error": f"Submission rejected: {'; '.join(errors)}",
                            "required_next_step": "Retrieve a better page set, shape it, and submit only once the requested scope is covered.",
                        }),
                    })
                    continue
                chosen = source_ids or select_balanced_narrative_ids(current_candidates, ask_obj)
                artifact["finalQueries"] = [final_sql] if final_sql else []
                artifact["selected_tickers"] = list(dict.fromkeys(selected_tickers))
                artifact["narrativeIds"] = list(dict.fromkeys(chosen))
                ensure_narrative_extractions(artifact, ask_obj, ask_text, artifact["narrativeIds"])
                artifact["resultSummary"] = f"{title}: {len(artifact['narrativeIds'])} pages"
                if artifact["narrativeExtractions"]:
                    artifact["resultSummary"] += f", {len(artifact['narrativeExtractions'])} extractions"
                messages.append({"role": "tool", "tool_call_id": tc.id, "content": json.dumps({"ok": True, "title": title})})
                artifact["_debug"] = {
                    "sql_queries": sql_queries,
                    "message_history": messages,
                    "system_prompt": system,
                    "input_tokens": input_tokens,
                    "output_tokens": output_tokens,
                    "current_candidates": current_candidates,
                    "candidate_history": candidate_history,
                }
                return artifact

            messages.append({"role": "tool", "tool_call_id": tc.id, "content": json.dumps({"error": f"Unknown tool: {name}"})})

        if iteration < MAX_LANE_ITERATIONS - 1:
            messages.append({"role": "user", "content": "Run at most one more retrieval or shaping step, then submit only if the candidate set is valid."})

    # Auto-submit from accumulated inventory + current candidates on turn limit
    if current_candidates:
        errors, _ = validate_submit(table, ask_obj, last_sql, last_rows, current_candidates, [])
        if not errors:
            chosen = select_balanced_narrative_ids(current_candidates, ask_obj)
            artifact["finalQueries"] = [last_sql] if last_sql else []
            artifact["narrativeIds"] = list(dict.fromkeys(chosen))
            ensure_narrative_extractions(artifact, ask_obj, ask_text, artifact["narrativeIds"])
            artifact["resultSummary"] = f"auto_submit_turn_limit: {len(artifact['narrativeIds'])} pages"
            if artifact["narrativeExtractions"]:
                artifact["resultSummary"] += f", {len(artifact['narrativeExtractions'])} extractions"
        else:
            artifact["resultSummary"] = f"lane_failed_no_valid_submit_after_{MAX_LANE_ITERATIONS}_iterations"
    else:
        artifact["resultSummary"] = f"lane_failed_no_valid_submit_after_{MAX_LANE_ITERATIONS}_iterations"
    artifact["_debug"] = {
        "sql_queries": sql_queries,
        "message_history": messages,
        "system_prompt": system,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "current_candidates": current_candidates,
        "candidate_history": candidate_history,
    }
    return artifact

In [ ]:
# ── §2.14 Lane dispatcher and inter-lane dependencies ─────────────
#
#  run_lane() routes each ask to the right runner (structured or narrative).
#  resolve_pipeline_ask() implements the SELECTOR-FIRST pattern:
#    if a previous lane explicitly exposed selected_tickers (e.g. 'top 5 banks'),
#    the current ask's tickers are rewritten from that signal — but ONLY from explicit
#    metadata, never by inferring from query prose or ranking arbitrary rows.
#    This keeps downstream retrieval scoped only when selection was made explicit.
#  In:  ask_obj + completed artifacts  |  Out: artifact from structured or narrative runner

def run_lane(ask_obj: dict, variable: str = "q1") -> dict:
    if client is None:
        raise RuntimeError("DeepSeek client unavailable. Set DEEPSEEK_API_KEY in §0.2.")
    # Short-circuit: tickers were in the router plan but none resolved to coverage universe.
    if ask_obj.get("_out_of_universe"):
        unresolved = ask_obj["_out_of_universe"]
        return {
            "variable": variable,
            "table": ask_obj.get("table", ""),
            "ask": ask_obj.get("ask", ""),
            "ask_object": ask_obj,
            "finalQueries": [],
            "resultSummary": f"OUT_OF_UNIVERSE: {', '.join(unresolved)} not in coverage universe — no data retrieved",
            "analysisAttachment": None,
            "provenanceIds": [],
            "narrativeIds": [],
            "narrativeExtractions": [],
            "_out_of_universe": unresolved,
        }
    if ask_obj.get("_missing_scope"):
        return {
            "variable": variable,
            "table": ask_obj.get("table", ""),
            "ask": ask_obj.get("ask", ""),
            "ask_object": ask_obj,
            "finalQueries": [],
            "resultSummary": "MISSING_SCOPE: company/ticker or industry scope required — no data retrieved",
            "analysisAttachment": None,
            "provenanceIds": [],
            "narrativeIds": [],
            "narrativeExtractions": [],
            "_missing_scope": ask_obj.get("_missing_scope"),
        }
    if ask_obj["table"] == "financial_documents_new":
        return run_narrative_lane(ask_obj, variable=variable)
    return run_structured_lane(ask_obj, variable=variable)


def _artifact_selected_tickers(artifact: dict) -> list[str]:
    selected = artifact.get("selected_tickers")
    if selected is None:
        selected = (artifact.get("selection") or {}).get("tickers")
    if selected is None:
        selected = (artifact.get("analysisAttachment") or {}).get("selected_tickers")
    if not isinstance(selected, list):
        return []
    out = []
    for ticker in selected:
        clean = str(ticker or "").strip().upper()
        if clean:
            out.append(clean)
    out = list(dict.fromkeys(out))
    requested = _unique_nonempty([str(t).upper() for t in (artifact.get("ask_object") or {}).get("tickers", [])])
    if requested and set(out) == set(requested):
        return []
    return out


def explicit_selected_tickers_from_artifacts(artifacts: list) -> list[str]:
    for artifact in reversed(artifacts):
        tickers = _artifact_selected_tickers(artifact)
        if tickers:
            return tickers
    return []


def resolve_pipeline_ask(query: str, ask: dict, completed_artifacts: list) -> dict:
    resolved = dict(ask)
    if ask.get("table") == "financial_documents_new" and completed_artifacts:
        selected_tickers = explicit_selected_tickers_from_artifacts(completed_artifacts)
        if selected_tickers:
            resolved["tickers"] = selected_tickers
            resolved["dependency"] = "selected_tickers_from_explicit_artifact"
    return resolved


print("Lane runners ready.")


In [ ]:
# ── §2.15 Run all lanes in sequence ───────────────────────────────────────────
# The notebook runs lanes sequentially so each lane workspace is easy to inspect.

if "plan" not in globals():
    raise RuntimeError("No router plan found. Run §1 first.")

lane_artifacts = []
for i, raw_ask in enumerate(plan, 1):
    ask = resolve_pipeline_ask(QUERY, raw_ask, lane_artifacts)
    print(f"\n{'='*60}")
    print(f"Running lane q{i}: {format_ask_line(ask)}")
    print('='*60)

    artifact = run_lane(ask, variable=f"q{i}")
    lane_artifacts.append(artifact)

    print(f"\nResult: {artifact['resultSummary']}")
    if artifact.get("analysisAttachment"):
        print("  → has analysis attachment (structured table)")
    if artifact.get("narrativeIds"):
        print(f"  → {len(artifact['narrativeIds'])} narrative source IDs")
    if artifact.get("narrativeExtractions"):
        print(f"  → {len(artifact['narrativeExtractions'])} extractions")
    if artifact.get("provenanceIds"):
        print(f"  → {len(artifact['provenanceIds'])} provenance IDs")

print(f"\n✓ §2 done: lane_artifacts has {len(lane_artifacts)} artifact(s). ")
print("Next: run §3.1 to attach provenanceIds, then §4.1–§4.4.")

In [ ]:
# ── §2.16 Inspect all lanes (full trace)  [OPTIONAL — debug/inspection only] ──
# Prints full SQL queries and assistant-turn sequence for each lane.
import os, json as _json
os.makedirs("outputs", exist_ok=True)

for lane in lane_artifacts:
    debug = lane.get("_debug", {})
    print(f"\n{'='*70}\n{lane['variable']} [{lane['table']}] — {lane['resultSummary']}")
    print("Ask:", _json.dumps(lane.get("ask_object", {}), indent=2))
    history = debug.get("message_history", [])
    print(f"Turns: {len(history)}\n")

    # Sequence the message history: for each assistant turn, print its content + tool calls + tool results
    for idx, msg in enumerate(history):
        role = msg.get("role", "?")
        if role == "system":
            continue  # skip system prompt dump
        if role == "user":
            content = msg.get("content", "")
            if content and len(content) < 400:
                print(f"[user {idx}] {content}")
            continue
        if role == "assistant":
            content = (msg.get("content") or "").strip()
            if content:
                print(f"[assistant {idx}] CONTENT: {content[:800]}")
            for tc in (msg.get("tool_calls") or []):
                fn = tc.get("function", {})
                name = fn.get("name", "")
                args_raw = fn.get("arguments", "")
                try:
                    args = _json.loads(args_raw)
                except Exception:
                    args = {"_raw": args_raw[:400]}
                if name == "run_sql":
                    sql = str(args.get("sql", "")).strip()
                    print(f"[assistant {idx}] run_sql:\n    {sql}")
                elif name == "submit":
                    print(f"[assistant {idx}] submit: {_json.dumps(args)[:600]}")
                else:
                    print(f"[assistant {idx}] {name}: {_json.dumps(args)[:400]}")
            continue
        if role == "tool":
            content = msg.get("content", "")
            try:
                parsed = _json.loads(content)
                # Compact single-line summary for tool responses
                if isinstance(parsed, dict):
                    summary = {}
                    for k in ("row_count", "columns", "candidate_count", "error", "required_next_step", "ok", "title", "tool", "candidates_remaining", "extraction_count"):
                        if k in parsed:
                            summary[k] = parsed[k]
                    print(f"[tool {idx}] {_json.dumps(summary)[:400]}")
                else:
                    print(f"[tool {idx}] {content[:300]}")
            except Exception:
                print(f"[tool {idx}] {content[:300]}")

    # Save full history to outputs/
    fn = f"outputs/lane_{lane['variable']}_{lane['table']}.json"
    with open(fn, 'w') as f:
        _json.dump({
            "ask": lane.get("ask_object", {}),
            "resultSummary": lane["resultSummary"],
            "sql_queries": debug.get("sql_queries", []),
            "message_history": history,
        }, f, indent=2)
    print(f"  → saved {fn}")


---
## §3 — Post-processing  ·  `runtime.ts`

**In:** `lane_artifacts`  →  **Out:** `lane_artifacts` updated in-place with `provenanceIds`

No LLM. **Run §3.1** once after §2 completes.

In [ ]:
# ── §3.1  Recover provenance IDs from structured lanes ────────────────────────
#
#  Runs deterministic provenance recovery across every structured artifact.
#  No LLM involved — walks each artifact's rows, calls recover_structured_provenance()
#  (§2.10b), and attaches provenanceIds in-place to the artifact.
#  market_data is exempt (source is the exchange; analyst states the date).
#  In:  lane_artifacts (list, mutated in-place)  |  Out: provenanceIds attached per artifact

# Fallback pass after lane execution. Structured lanes now try to recover
# provenance during each successful SQL step; this cell fills any gaps from the
# final submitted attachment.

for artifact in lane_artifacts:
    table = artifact["table"]
    if table not in {"financial_statements", "ratios", "management_comp"}:
        print(f"[{artifact['variable']}] {table} — {len(artifact.get('provenanceIds', []))} provenance IDs")
        continue
    if artifact.get("provenanceIds"):
        print(f"[{artifact['variable']}] {table} — {len(artifact['provenanceIds'])} provenance IDs")
        continue
    att = artifact.get("analysisAttachment")
    if not att:
        print(f"[{artifact['variable']}] {table} — no analysisAttachment, skipping")
        continue
    final_rows = [dict(zip(att["headers"], r)) for r in att["rows"]]
    ids = recover_structured_provenance(table, final_rows)
    artifact["provenanceIds"] = ids
    print(f"[{artifact['variable']}] {table} — recovered {len(ids)} provenance IDs")

print("\n✓ §3 done: provenanceIds attached to all structured artifacts. Next: run §4.1–§4.4.")

---
## §4 — Analyst  ·  `analyst.ts`

**In:** `lane_artifacts` + narrative sources  →  **Out:** `analyst_result`

No tools — analyst synthesises from pre-computed evidence only.
§4.1–§4.4 run in order. **§4.3 produces `analyst_result`; §4.4 displays it.** §4.5 is `[OPTIONAL]`.

In [ ]:
# ── §4.1  Assemble analyst sources ───────────────────────────────────────────
#
#  The analyst receives narrative evidence only — structured tables travel separately.
#  assemble_analyst_sources() selects a balanced set of narrative page IDs across lanes,
#  fetches their text from D1, and caps total chars to stay within the analyst context window.
#  Structured provenance IDs are kept separate for source-panel display, not analyst context.
#  In:  lane_artifacts  |  Out: sources list [{id, ticker, filing, text, …}]

# AnalystContext gets narrative evidence only. Structured provenance stays separate.

MAX_ANALYST_SOURCE_TOTAL_CHARS = 50_000
MAX_ANALYST_SOURCE_CHARS_PER_PAGE = 2_500
MAX_ANALYST_SOURCE_IDS = 24
FETCH_DOC_CHUNK_SIZE = 80


def fetch_document_rows_by_ids(ids: list[int], columns: str) -> dict[int, dict]:
    row_map = {}
    clean_ids = list(dict.fromkeys(int(i) for i in ids if int(i) > 0))
    for start in range(0, len(clean_ids), FETCH_DOC_CHUNK_SIZE):
        chunk = clean_ids[start:start + FETCH_DOC_CHUNK_SIZE]
        ph = ",".join(["?"] * len(chunk))
        res = run_sql(f"SELECT {columns} FROM financial_documents_new WHERE id IN ({ph})", chunk)
        if not res.get("success"):
            print(f"Source fetch failed for chunk {start // FETCH_DOC_CHUNK_SIZE + 1}: {res.get('error')}")
            continue
        for row in res.get("rows", []):
            row_map[int(row["id"])] = row
    return row_map


def _select_analyst_source_ids_by_lane(artifacts: list) -> list[int]:
    narrative_lanes = [art for art in artifacts if art.get("narrativeIds")]
    if not narrative_lanes:
        return []
    selected, seen = [], set()
    base_quota = max(1, MAX_ANALYST_SOURCE_IDS // len(narrative_lanes))

    lane_ids = []
    for art in narrative_lanes:
        ids = [int(i) for i in art.get("narrativeIds", []) if int(i) > 0]
        lane_ids.append(ids)
        for pid in ids[:base_quota]:
            if pid not in seen and len(selected) < MAX_ANALYST_SOURCE_IDS:
                selected.append(pid)
                seen.add(pid)

    # Fill any remaining capacity round-robin so later lanes are not starved.
    offset = base_quota
    while len(selected) < MAX_ANALYST_SOURCE_IDS:
        added = False
        for ids in lane_ids:
            if offset < len(ids):
                pid = ids[offset]
                if pid not in seen:
                    selected.append(pid)
                    seen.add(pid)
                    added = True
                    if len(selected) >= MAX_ANALYST_SOURCE_IDS:
                        break
        if not added:
            break
        offset += 1
    return selected


def assemble_analyst_sources(artifacts: list) -> list:
    extraction_map = {}
    provenance_display_ids = []
    for art in artifacts:
        provenance_display_ids.extend(art.get("provenanceIds", []))
        for ext in art.get("narrativeExtractions", []):
            pid = int(ext["page_id"])
            extraction_map.setdefault(pid, str(ext["extracted_text"]).strip())

    selected_ids = _select_analyst_source_ids_by_lane(artifacts)
    if not selected_ids:
        print(f"Analyst sources: 0 pages. Provenance recovered separately: {len(list(dict.fromkeys(provenance_display_ids)))}")
        return []

    row_map = fetch_document_rows_by_ids(
        selected_ids,
        "id, ticker, filing_period, filing_type, filing_year, pg, jpg_path, text, search_text",
    )

    sources, total = [], 0
    for tag_idx, pid in enumerate(selected_ids, 1):
        if total >= MAX_ANALYST_SOURCE_TOTAL_CHARS:
            break
        row = row_map.get(pid, {})
        cap = min(MAX_ANALYST_SOURCE_CHARS_PER_PAGE, MAX_ANALYST_SOURCE_TOTAL_CHARS - total)
        extracted = extraction_map.get(pid, "")
        raw = str(row.get("text") or row.get("search_text") or "")
        text = (extracted or raw)[:cap]
        if not text:
            continue
        sources.append({
            "id": pid,
            "tag": str(tag_idx),
            "ticker": row.get("ticker", ""),
            "filing_period": row.get("filing_period", ""),
            "filing_type": row.get("filing_type", ""),
            "filing_year": row.get("filing_year", ""),
            "page_number": row.get("pg", 0),
            "jpg_path": row.get("jpg_path", ""),
            "text": text,
            "content_kind": "extraction" if extracted else "raw_page",
        })
        total += len(text)

    extraction_count = sum(1 for s in sources if s["content_kind"] == "extraction")
    provenance_count = len(list(dict.fromkeys(provenance_display_ids)))
    print(f"Analyst sources: {len(sources)} pages ({extraction_count} extractions, {len(sources) - extraction_count} raw). Provenance recovered separately: {provenance_count}. ~{total:,} chars")
    return sources


analyst_sources = assemble_analyst_sources(lane_artifacts)


In [ ]:
# ── §4.1b  Assemble source panel payload ─────────────────────────────────────
#
#  Builds the source-panel payload for the frontend — thumbnails, PDF links, page numbers.
#  Combines narrative sources (from §4.1) with structured provenanceIds (from §3.1).
#  Not consumed by the analyst LLM — purely for the user-facing citation display.
#  In:  artifacts + sources  |  Out: source_panel_payload (dict sent to frontend)

# Mirrors what the frontend source panel renders: every page surfaced by any
# lane (narrative submissions ∪ structured provenance), deduped, with metadata.

MAX_SOURCE_PANEL_STRUCTURED_IDS = 60
MAX_SOURCE_PANEL_STRUCTURED_IDS_PER_TICKER = 3


def _select_structured_panel_ids(structured_ids: list[int], narrative_set: set[int], row_map: dict) -> tuple[list[int], int]:
    structured_only = [int(i) for i in dict.fromkeys(structured_ids) if int(i) > 0 and int(i) not in narrative_set]
    if len(structured_only) <= MAX_SOURCE_PANEL_STRUCTURED_IDS:
        return structured_only, 0

    buckets: dict[str, list[int]] = {}
    ticker_order = []
    for pid in structured_only:
        row = row_map.get(pid, {})
        ticker = str(row.get("ticker") or "UNKNOWN").upper()
        if ticker not in buckets:
            buckets[ticker] = []
            ticker_order.append(ticker)
        if len(buckets[ticker]) < MAX_SOURCE_PANEL_STRUCTURED_IDS_PER_TICKER:
            buckets[ticker].append(pid)

    selected = []
    round_idx = 0
    while len(selected) < MAX_SOURCE_PANEL_STRUCTURED_IDS:
        added = False
        for ticker in ticker_order:
            bucket = buckets.get(ticker, [])
            if round_idx < len(bucket):
                selected.append(bucket[round_idx])
                added = True
                if len(selected) >= MAX_SOURCE_PANEL_STRUCTURED_IDS:
                    break
        if not added:
            break
        round_idx += 1

    if len(selected) < MAX_SOURCE_PANEL_STRUCTURED_IDS:
        seen = set(selected)
        for pid in structured_only:
            if pid in seen:
                continue
            selected.append(pid)
            seen.add(pid)
            if len(selected) >= MAX_SOURCE_PANEL_STRUCTURED_IDS:
                break

    return selected, max(0, len(structured_only) - len(selected))


def assemble_source_panel(artifacts: list) -> list:
    narrative_ids = []
    structured_ids = []
    for art in artifacts:
        narrative_ids.extend(art.get("narrativeIds", []))
        structured_ids.extend(art.get("provenanceIds", []))

    narrative_ids = [int(i) for i in dict.fromkeys(narrative_ids) if int(i) > 0]
    structured_ids = [int(i) for i in dict.fromkeys(structured_ids) if int(i) > 0]
    narrative_set = set(narrative_ids)
    all_lookup_ids = list(dict.fromkeys(narrative_ids + structured_ids))
    if not all_lookup_ids:
        print("Source panel: 0 pages.")
        return []

    row_map = fetch_document_rows_by_ids(
        all_lookup_ids,
        "id, ticker, filing_period, filing_type, filing_year, pg, jpg_path",
    )
    structured_display_ids, structured_omitted = _select_structured_panel_ids(structured_ids, narrative_set, row_map)
    panel_ids = list(dict.fromkeys(narrative_ids + structured_display_ids))
    structured_display_set = set(structured_display_ids)
    structured_set = set(structured_ids)

    panel = []
    for pid in panel_ids:
        row = row_map.get(pid)
        if not row:
            continue
        in_narr = pid in narrative_set
        in_struct = pid in structured_set
        kind = "both" if (in_narr and in_struct) else ("narrative" if in_narr else "structured")
        panel.append({
            "id": pid,
            "ticker": row.get("ticker", ""),
            "filing_period": row.get("filing_period", ""),
            "filing_type": row.get("filing_type", ""),
            "filing_year": row.get("filing_year", ""),
            "page_number": row.get("pg", 0),
            "jpg_path": row.get("jpg_path", ""),
            "kind": kind,
            "structured_provenance_total": len([i for i in structured_ids if i not in narrative_set]),
            "structured_provenance_omitted": structured_omitted,
        })

    panel.sort(key=lambda r: (str(r["ticker"] or ""), str(r["filing_period"] or ""), int(r["page_number"] or 0)))
    n_narr = sum(1 for p in panel if p["kind"] == "narrative")
    n_struct = sum(1 for p in panel if p["kind"] == "structured")
    n_both = sum(1 for p in panel if p["kind"] == "both")
    omitted_msg = f", {structured_omitted} structured omitted from panel" if structured_omitted else ""
    print(f"Source panel: {len(panel)} pages ({n_narr} narrative, {n_struct} structured, {n_both} both{omitted_msg})")
    return panel


source_panel = assemble_source_panel(lane_artifacts)


In [ ]:
# ── §4.2  Build the analyst user message ─────────────────────────────────────
#
#  Assembles the analyst's user message — the full evidence package it will synthesize from:
#    EXECUTION ARTIFACTS  — typed ask objects + result summaries + final SQL (traceability)
#    COMPUTED DATA        — structured table IDs + CSV rows with coverage metadata
#    SOURCES              — selected narrative page extractions (capped, balanced by lane)
#  Context release rule: analyst does NOT see schema docs, other table details, or
#  unselected page text. It synthesizes only from the supplied evidence.
#  In:  QUERY + artifacts + sources  |  Out: analyst user message string

def build_analyst_message(query: str, artifacts: list, sources: list) -> str:
    computed_tables = [art.get("analysisAttachment") for art in artifacts if art.get("analysisAttachment")]
    coverage_lines = []
    for i, t in enumerate(computed_tables, 1):
        headers = t.get("headers", [])
        rows = t.get("rows", [])
        coverage = t.get("coverage", {})
        coverage_summary = _coverage_summary_text(coverage)
        base = f"[TABLE {i}] {t.get('title', '')}: {len(rows)} rows; columns: {', '.join(map(str, headers))}"
        if coverage_summary:
            base += f"; coverage: {coverage_summary}"
        coverage_lines.append(base)
    for s in sources:
        period = s.get("filing_period") or f"{s.get('filing_type', '')} {s.get('filing_year', '')}"
        kind = "extracted evidence" if s.get("content_kind") == "extraction" else "source page"
        coverage_lines.append(f"[{s['tag']}] {s.get('ticker', '')} {period} p{s.get('page_number', '')}: {kind}")
    coverage_section = "CONTEXT COVERAGE\n" + ("\n".join(coverage_lines) if coverage_lines else "none")

    if artifacts:
        exec_lines = []
        oou_lines = []
        missing_scope_lines = []
        for i, art in enumerate(artifacts, 1):
            summary = art.get("resultSummary", "")
            selected = art.get("selected_tickers") or []
            selected_line = f"\n   Selected tickers: {', '.join(selected)}" if selected else ""
            exec_lines.append(f"{i}. [{art.get('table', '')}] {art.get('ask', '')}\n   Result: {summary}{selected_line}")
            if art.get("_out_of_universe"):
                unresolved = art["_out_of_universe"]
                oou_lines.append(
                    f"  - {', '.join(unresolved)}: not in the coverage universe. "
                    "Acknowledge this in your answer — do not attempt to invent data for these tickers."
                )
            # _unresolved_tickers lives on ask_object (set by normalize_router_ask)
            _partial_oou = (
                art.get("_unresolved_tickers")
                or (art.get("ask_object") or {}).get("_unresolved_tickers")
            )
            if _partial_oou:
                oou_lines.append(
                    f"  - {', '.join(_partial_oou)}: named by the user but not found in the coverage universe. "
                    "Disclose that no data was retrieved for these tickers."
                )
            if art.get("_missing_scope"):
                missing_scope_lines.append(
                    f"  - {art.get('ask', '')}: missing company/ticker or industry scope. "
                    "Ask the user for the missing scope instead of guessing."
                )
        exec_section = "RETRIEVAL SUMMARY\n" + "\n\n".join(exec_lines)
        if oou_lines:
            exec_section += "\n\nOUT OF COVERAGE\n" + "\n".join(oou_lines)
        if missing_scope_lines:
            exec_section += "\n\nMISSING SCOPE\n" + "\n".join(missing_scope_lines)
    else:
        exec_section = "RETRIEVAL SUMMARY\nnone"

    if computed_tables:
        table_lines = [f"[TABLE {i}] {t['title']}\n{t['csv']}" for i, t in enumerate(computed_tables, 1)]
        table_section = "COMPUTED DATA\n" + "\n\n".join(table_lines)
    else:
        table_section = "COMPUTED DATA\nnone"

    if sources:
        source_lines = []
        for s in sources:
            period = s.get("filing_period") or f"{s.get('filing_type', '')} {s.get('filing_year', '')}"
            header = f"══ [{s['tag']}] {s['ticker']} {period} p{s['page_number']}"
            source_lines.append(f"{header}\n{s['text']}")
        source_section = "SOURCES\n" + "\n\n---\n\n".join(source_lines)
    else:
        source_section = "SOURCES\nnone"

    return "\n\n".join([f"USER QUERY\n{query}", coverage_section, exec_section, table_section, source_section])


analyst_user_msg = build_analyst_message(QUERY, lane_artifacts, analyst_sources)
print(f"Analyst user message: {len(analyst_user_msg):,} chars")


In [ ]:
# ── §4.3  Run the analyst ─────────────────────────────────────────────────────
#
#  Runs the analyst LLM (DeepSeek V4 Pro) — no tools, one call, pure synthesis.
#  System: analyst/prompt.md (citation rules, unit conventions, output markers).
#  The analyst may build labelled illustrative scenarios from supplied tables but
#  must NOT invent inputs as if sourced, treat SQL as evidence, or recompute
#  historicals that lanes already provided.
#  In:  ROUTER_SYSTEM (unused) + analyst user message (from §4.2)
#  Out: analyst_result {answer, cited_sources, cited_tables, user_message, …}

# No tools — analyst only synthesises from pre-computed evidence.

USER_MEMORIES = []   # ["User prefers PKR billions", "User is a buy-side analyst"]


def _valid_table_tags(artifacts: list) -> set[str]:
    return {str(i) for i, art in enumerate([a for a in artifacts if a.get("analysisAttachment")], 1)}


def _valid_source_tags(sources: list) -> set[str]:
    return {str(s.get("tag")) for s in sources}


def _sanitize_analyst_citations(answer: str, valid_tables: set[str], valid_sources: set[str]) -> str:
    # Table IDs are internal provenance markers. Keep USED_TABLE_TAGS for audit,
    # but remove visible [TABLE N] artifacts from the user-facing answer.
    clean = re.sub(r"\s*\[TABLE\s+\d+\]\s*", " ", answer, flags=re.I)
    clean = re.sub(r"\n{3,}", "\n\n", clean)
    return clean.strip()


def run_analyst(query: str, artifacts: list, sources: list, user_memories: list = None) -> dict:
    valid_tables = _valid_table_tags(artifacts)
    valid_sources = _valid_source_tags(sources)
    citation_contract = (
        "\n\nCITATION CONTRACT\n"
        f"- Valid computed table IDs for USED_TABLE_TAGS: {sorted(valid_tables, key=int) if valid_tables else []}. Do not write visible [TABLE N] citations in the answer.\n"
        f"- Valid source tags: {sorted(valid_sources, key=int) if valid_sources else []}. Do not cite any other [N].\n"
        "- Raw source text may contain Markdown tables; those are source page content, not computed data tables.\n"
    )
    if user_memories:
        mem_lines = "\n".join(f"- {m}" for m in user_memories)
        system = ANALYST_PROMPT_MD + citation_contract + f"\n\nUSER MEMORY\n{mem_lines}"
    else:
        system = ANALYST_PROMPT_MD + citation_contract

    messages = [{"role": "system", "content": system}]
    user_message = build_analyst_message(query, artifacts, sources)
    messages.append({"role": "user", "content": user_message})

    if analyst_client is None:
        raise RuntimeError("DeepSeek analyst client unavailable. Set DEEPSEEK_API_KEY in §0.2.")

    response = analyst_client.chat.completions.create(
        model=MODEL_ANALYST,
        messages=messages,
        temperature=0.1,
        stream=False,
    )
    raw = (response.choices[0].message.content or "").strip()

    def extract_tags(text, marker):
        pattern = re.compile(rf'\n?{marker}:\s*(\[[^\n]*\])\s*$', re.IGNORECASE)
        match = pattern.search(text)
        if not match:
            return text.strip(), []
        cleaned = text[:match.start()].strip()
        try:
            tags = json.loads(match.group(1))
            return cleaned, [str(t) for t in tags if str(t).strip()]
        except Exception:
            return cleaned, []

    answer_after_tables, used_table_tags = extract_tags(raw, "USED_TABLE_TAGS")
    clean_answer, used_source_tags = extract_tags(answer_after_tables, "USED_SOURCE_TAGS")
    used_table_tags = [t for t in used_table_tags if t in valid_tables]
    used_source_tags = [t for t in used_source_tags if t in valid_sources]
    clean_answer = _sanitize_analyst_citations(clean_answer, valid_tables, valid_sources)

    disclaimer = "\n\n---\n*Disclaimer: This is not financial advice. psxGPT can make mistakes. Please verify sources and do your own due diligence.*"
    return {
        "answer": clean_answer + disclaimer,
        "used_sources": used_source_tags,
        "used_tables": used_table_tags,
        "system_prompt": system,
        "user_message": user_message,
        "raw_answer": raw,
        "usage": {"input_tokens": response.usage.prompt_tokens, "output_tokens": response.usage.completion_tokens},
    }


analyst_result = run_analyst(QUERY, lane_artifacts, analyst_sources, user_memories=USER_MEMORIES)
print(f"Analyst complete. Tokens: {analyst_result['usage']['input_tokens']} in / {analyst_result['usage']['output_tokens']} out")
print(f"Cited sources: {analyst_result['used_sources']}")
print(f"Cited tables:  {analyst_result['used_tables']}")

# ✓ Checkpoint — §4 complete
print("\n✓ §4 done: analyst_result ready. Run §4.4 to display the answer.")

In [ ]:
# ── §4.4  Display the answer ───────────────────────────────────────────────────
ans = analyst_result.get("answer", "") if isinstance(analyst_result, dict) else ""
if ans:
    display(Markdown(ans))
else:
    print("No analyst answer available.")


In [ ]:
# ── §4.5  Inspect analyst context  [OPTIONAL — debug/inspection only] ──────────
# See the full user message the analyst received — useful when citations are wrong
# or the answer is missing something you expected to be in the evidence.

print(analyst_result["user_message"][:8000])  # first 8k chars

---
## §5 — End-to-end  ·  `orchestrator.ts`

Runs all stages with a single query. Edit `QUERY_E2E` in §5 and run.
Use §1–§4 to step through and inspect individual stages; use §5 for a clean end-to-end run.

**§6 vs §5:** §6 (`inspect_pipeline`) also runs end-to-end but prints every evidence layer for debugging.

In [ ]:
# ── §5  Full pipeline ─────────────────────────────────────────────────────────
#
#  End-to-end pipeline runner — use this for quick queries once §0 is loaded.
#  Runs: run_router → run_lane×N (sequential, selector-first) → provenance recovery
#         → assemble sources → build analyst message → run_analyst → display.
#  For first-time setup or credential issues, run §0 cells individually first.
#  In:  QUERY (set below)  |  Out: analyst_result displayed inline

# Edit QUERY_E2E and run. Prints a summary trace then renders the answer.

QUERY_E2E = "Compare ROE and net margin for HBL and MCB over the last 3 years"
MEMORIES_E2E = []

print(f"Query: {QUERY_E2E}")
print("=" * 60)

print("\n[Stage 1] Router...")
e2e_router = run_router(QUERY_E2E)
e2e_plan = e2e_router["plan"]
for i, ask in enumerate(e2e_plan, 1):
    print(f"  Ask {i}: {format_ask_line(ask)}")

print(f"\n[Stage 2] Running {len(e2e_plan)} lane(s)...")
e2e_artifacts = []
for i, raw_ask in enumerate(e2e_plan, 1):
    ask = resolve_pipeline_ask(QUERY_E2E, raw_ask, e2e_artifacts)
    artifact = run_lane(ask, variable=f"q{i}")
    e2e_artifacts.append(artifact)
    print(f"  q{i} [{ask['table']}]: {artifact['resultSummary']}")

print("\n[Stage 3] Post-processing provenance...")
for art in e2e_artifacts:
    table = art["table"]
    if table in {"financial_statements", "ratios", "management_comp"} and not art.get("provenanceIds"):
        att = art.get("analysisAttachment")
        if att:
            final_rows = [dict(zip(att["headers"], r)) for r in att["rows"]]
            art["provenanceIds"] = recover_structured_provenance(table, final_rows)
            print(f"  [{art['variable']}] {table} — {len(art['provenanceIds'])} provenance IDs")

print("\n[Stage 4] Assembling sources and running analyst...")
e2e_sources = assemble_analyst_sources(e2e_artifacts)
e2e_analyst = run_analyst(QUERY_E2E, e2e_artifacts, e2e_sources, user_memories=MEMORIES_E2E)

lane_in = sum(a.get("_debug", {}).get("input_tokens", 0) for a in e2e_artifacts)
lane_out = sum(a.get("_debug", {}).get("output_tokens", 0) for a in e2e_artifacts)
print("\n--- Token usage ---")
print(f"  Router:  {e2e_router['usage']['input_tokens']} in / {e2e_router['usage']['output_tokens']} out")
print(f"  Lanes:   {lane_in} in / {lane_out} out")
print(f"  Analyst: {e2e_analyst['usage']['input_tokens']} in / {e2e_analyst['usage']['output_tokens']} out")

print("\n--- Answer ---")
display(Markdown(e2e_analyst.get("answer", "No answer")))


# --- Full lane trace for e2e (written to outputs/) ---
import os as _os, json as _json
_os.makedirs("outputs", exist_ok=True)
for _lane in e2e_artifacts:
    _debug = _lane.get("_debug", {})
    _fn = f"outputs/e2e_lane_{_lane['variable']}_{_lane['table']}.json"
    with open(_fn, 'w') as _f:
        _json.dump({
            "ask": _lane.get("ask_object", {}),
            "resultSummary": _lane["resultSummary"],
            "sql_queries": _debug.get("sql_queries", []),
            "message_history": _debug.get("message_history", []),
        }, _f, indent=2)
    print(f"Saved {_fn}")

# ✓ Checkpoint — §5 complete
print("\n✓ §5 done: full pipeline complete. Answer rendered above; lane traces saved to outputs/.")
print("  Use §6 (Cell 44) to inspect every evidence layer in detail.")

In [ ]:
# ── §6  Pipeline inspector ───────────────────────────────────────────────────
#
#  Pipeline inspector — pretty-prints the full lane_artifacts and analyst_result
#  after a pipeline run. Useful for debugging routing decisions, coverage gaps,
#  and analyst citation accuracy without re-running the full pipeline.
#  In:  lane_artifacts + analyst_result (from §5 run)  |  Out: formatted debug output

# inspect_pipeline(query) runs the full pipeline and prints every layer:
#   - Router plan (asks)
#   - Per-lane artifact summary (counts by kind)
#   - Full source panel payload (every page the frontend would render)
#   - The 12 sources actually fed into AnalystContext
#   - Which tags the analyst cited (USED_SOURCE_TAGS / USED_TABLE_TAGS)
#   - Diff: panel pages beyond cap; fed-but-not-cited tags
# Use this to see whether useful pages get dropped between
# lane → panel → analyst → citation.

def inspect_pipeline(query: str, memories: list = None) -> dict:
    memories = memories or []
    print(f"Query: {query}")
    print("=" * 78)

    print("\n[1] Router")
    router_out = run_router(query)
    plan = router_out["plan"]
    for i, ask in enumerate(plan, 1):
        print(f"  {i}. {format_ask_line(ask)}")

    print(f"\n[2] Lanes (n={len(plan)})")
    artifacts = []
    for i, raw_ask in enumerate(plan, 1):
        ask = resolve_pipeline_ask(query, raw_ask, artifacts)
        art = run_lane(ask, variable=f"q{i}")
        artifacts.append(art)
        nids = len(art.get("narrativeIds", []))
        pids = len(art.get("provenanceIds", []))
        tickers = ",".join(ask.get("tickers", []) or [])
        print(f"  q{i} [{ask['table']}] tickers={tickers} narrativeIds={nids} provenanceIds={pids}")
        print(f"     summary: {art.get('resultSummary', '')}")

    print("\n[3] Post-processing structured provenance")
    for art in artifacts:
        if art["table"] in {"financial_statements", "ratios", "management_comp"} and not art.get("provenanceIds"):
            att = art.get("analysisAttachment")
            if att:
                final_rows = [dict(zip(att["headers"], r)) for r in att["rows"]]
                art["provenanceIds"] = recover_structured_provenance(art["table"], final_rows)
                print(f"  [{art['variable']}] +{len(art['provenanceIds'])} structured provenance IDs")

    print("\n[4a] Source panel payload")
    panel = assemble_source_panel(artifacts)
    for p in panel:
        period = p.get("filing_period") or f"{p.get('filing_type','')} {p.get('filing_year','')}"
        print(f"  [{p['kind']:>10}] id={p['id']:<7} {p['ticker']:<6} {period:<10} p{p['page_number']}")

    print("\n[4b] AnalystContext (capped at MAX_SELECTED_SOURCE_IDS)")
    sources = assemble_analyst_sources(artifacts)
    for s in sources:
        period = s.get("filing_period") or f"{s.get('filing_type','')} {s.get('filing_year','')}"
        snippet = (s.get("text") or "").replace("\n", " ")[:100]
        print(f"  [{s['tag']:>3}] {s['content_kind']:<10} id={s.get('id','?'):<7} {s['ticker']:<6} {period:<10} p{s['page_number']}  {snippet}")

    print("\n[5] Analyst")
    analyst = run_analyst(query, artifacts, sources, user_memories=memories)
    cited = set(analyst["used_sources"])
    fed_tags = {s["tag"] for s in sources}
    not_cited = sorted(fed_tags - cited, key=lambda t: int(t) if str(t).isdigit() else 0)
    panel_beyond_cap = max(0, len(panel) - len(sources))
    fed_ids = {s.get("id") for s in sources if s.get("id") is not None}
    panel_dropped = [p for p in panel if p["id"] not in fed_ids]
    print(f"  USED_SOURCE_TAGS: {sorted(cited, key=lambda t: int(t) if str(t).isdigit() else 0)}")
    print(f"  USED_TABLE_TAGS:  {analyst['used_tables']}")
    print(f"  fed-but-not-cited: {not_cited}")
    print(f"  panel pages dropped from analyst: {len(panel_dropped)} of {len(panel)} (cap={MAX_SELECTED_SOURCE_IDS})")

    print("\n--- Answer ---")
    display(Markdown(analyst.get("answer", "No answer")))

    return {
        "router": router_out,
        "artifacts": artifacts,
        "panel": panel,
        "analyst_sources": sources,
        "analyst": analyst,
    }
